# **Smart Resume Generator: Full System Overview**

This project builds an AI-assisted system that helps optimize a resume for a specific job description while preserving the original structure and authenticity of the document. Instead of relying on a single prompt to rewrite a resume, the system follows a structured workflow that analyzes alignment between a resume and a job description, identifies gaps, generates controlled suggestions, and allows the user to decide which changes should be applied.

The core idea is to treat resume optimization as a structured pipeline rather than a one-step generation task. The system first converts both the resume and the job description into structured representations. It then measures semantic alignment between the two using embedding-based similarity and weighted keyword analysis. Based on this alignment, the system generates suggestions for improving specific sections such as the summary, skills, experience bullets, and project descriptions.

All suggestions are validated before being applied. The system enforces schema constraints, prevents hallucinated information, and requires user approval before modifying the resume. This ensures that the final output remains truthful, structured, and consistent with the original content.

The architecture is organized into layers. Each layer has a specific responsibility in the pipeline. Earlier layers focus on extracting and structuring information, while later layers focus on alignment analysis, controlled LLM generation, validation, and final document assembly.

Layer 1 extracts and structures resume information.
Layer 2 converts the job description into a weighted feature map.
Layer 3 performs semantic alignment and generates optimization suggestions.
Later layers apply validated changes and export the final resume in formats such as PDF, DOCX, or LaTeX.

By separating planning, generation, validation, and execution into distinct steps, the system makes the use of large language models more reliable, transparent, and controllable than traditional one-shot prompting.

# **Layer 1 Overview**

In Layer 1, I built the resume intake and structuring pipeline. The goal of this layer is to take a raw resume file, usually in PDF or DOCX format, extract the text as cleanly as possible, repair obvious parsing issues, and convert everything into a structured JSON format that later layers can use reliably.

This layer is important because everything after this depends on having consistent, structured resume data. If the extraction is messy here, then semantic matching, keyword alignment, and rewriting in later layers will also become messy.

So I designed Layer 1 to do three main things:

First, create a session-based user profile and persistence system.

Second, parse and clean raw resume text from uploaded files.

Third, structure that cleaned text into organized resume sections such as summary, education, skills, experience, projects, and certifications.

In [ ]:
!pip install email_validator PyPDF2
from pydantic import BaseModel, Field, EmailStr
from typing import List, Dict, Optional, Any
import uuid
import datetime
import json
import os

# **1. Session and Profile Initialization**

The first part defines a UserProfile schema using Pydantic. This acts like the base profile object for a user session.

It stores:

the session id
basic contact details
flat resume fields like summary, skills, education, experience, and projects
raw extracted content
and the job description placeholder for later layers

The reason I included both flat fields and an extracted raw copy is that I wanted two levels of representation.

The flat fields are easy to show quickly in the UI or debug prints.

The extracted dictionary is more useful for research, downstream processing, and richer resume reconstruction later.

I also added helper functions for saving and loading JSON files. This makes the pipeline persistent, so every session can be stored and revisited instead of existing only in memory.

So in simple terms, this part gives the system a reliable way to start a new user session, validate the schema, and save the user’s resume state to disk.

In [ ]:
class UserProfile(BaseModel):
    session_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    version: str = "1.0"
    created_at: str = Field(default_factory=lambda: datetime.datetime.utcnow().isoformat())

    # Personal Info
    name: Optional[str] = None
    email: Optional[EmailStr] = None
    phone: Optional[str] = None
    linkedin: Optional[str] = None
    portfolio: Optional[str] = None
    location: Optional[str] = None

    # Structured resume data from upload
    summary: List[str] = Field(default_factory=list)
    education: List[str] = Field(default_factory=list)
    skills: List[str] = Field(default_factory=list)
    experience: List[str] = Field(default_factory=list)
    projects: List[str] = Field(default_factory=list)
    certifications: List[str] = Field(default_factory=list)

    # NEW FLAT EXTRACTION OUTPUT
    contact_info: List[str] = Field(default_factory=list)
    any_links: List[str] = Field(default_factory=list)

    # Raw extraction complete copy (optional but useful)
    extracted: Dict[str, List[str]] = Field(default_factory=dict)

    # JD attachment (for later phases)
    job_description_raw: Optional[str] = None


In [ ]:
def ensure_dir_exists(path: str):
    """Ensure parent directory exists before writing files."""
    os.makedirs(os.path.dirname(path), exist_ok=True)

def save_json(obj: BaseModel | Dict[str, Any], path: str):
    ensure_dir_exists(path)
    data = obj.model_dump() if isinstance(obj, BaseModel) else obj
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

def load_json(path: str) -> Dict[str, Any]:
    with open(path) as f:
        return json.load(f)

def profile_path(session_id: str) -> str:
    return f"data/{session_id}.profile.json"

def featuremap_path(session_id: str) -> str:
    return f"data/{session_id}.jdmap.json"

def new_session() -> UserProfile:
    up = UserProfile()
    save_json(up, profile_path(up.session_id))
    print("🆕 Session created:", up.session_id)
    return up

In [ ]:
# Test A: Empty profile generation & schema validation
test_profile = new_session()
loaded = UserProfile(**load_json(profile_path(test_profile.session_id)))

assert loaded.session_id == test_profile.session_id
assert loaded.version == "1.0"
assert isinstance(loaded.education, list)
assert isinstance(loaded.experience, list)
assert isinstance(loaded.skills, list)

print("✅ Test A Passed: Schema + persistence OK.")

# **2. Resume Parsing and Text Cleaning**

The next part handles reading the uploaded resume itself.

I support both PDF and DOCX files.

For PDFs, I use a layered fallback strategy:

First I try pdfplumber, because it usually gives the cleanest native text.

If that fails, I fall back to PyPDF2.

If that still does not work well, I use OCR through Tesseract.

That matters because real resumes are inconsistent. Some PDFs are text-based, some are weirdly encoded, and some behave almost like images. So I did not want the extraction to fail just because one parser could not read the file.

For DOCX files, I read paragraphs directly using python-docx, and I preserve list-style bullets when possible.

After raw extraction, I normalize the text.

That includes:

removing non-ASCII artifacts
cleaning bullet symbols
collapsing extra spaces
preserving meaningful newlines

This is the stage where I try to turn messy resume text into something readable and consistent enough for section segmentation.

So this part is basically the text ingestion and cleanup engine.

In [ ]:
!apt-get update -qq
!apt-get install -y tesseract-ocr libtesseract-dev poppler-utils

# Core Python dependencies
!pip install -q python-docx pdfplumber PyPDF2 pytesseract Pillow huggingface-hub

# **3. Header Repair and Entity Recovery**

After extraction, I noticed that resume headers often break in small but important ways.

For example:

names may get split into separate capital letters
phone numbers may be inconsistently formatted
LinkedIn links may be partial or broken
some resumes may not even have a clear “Summary” heading

So I added a repair layer.

This layer tries to:

fix broken uppercase names
extract email addresses
recover and standardize phone numbers
extract links like LinkedIn or GitHub
rebuild the contact line if it is fragmented
and inject a SUMMARY heading if the resume clearly has a summary-like block but no heading

There is also an optional LLM repair step for the top header if the first line still looks broken even after heuristics.

I kept that repair very narrow on purpose. It only patches the name and contact line if needed, instead of letting the model rewrite the resume.

So this stage is really about making sure the top of the resume is usable before the structuring stage begins.

In [ ]:
import os, re, docx, pdfplumber, PyPDF2, pytesseract
from typing import List, Tuple, Optional
from huggingface_hub import InferenceClient

# ----------------------------
# Tunables (no need to change)
# ----------------------------
PHONE_OUTPUT_FORMAT = "ats"  # "ats" -> 559-457-9729, "national" -> (559) 457-9729, "asis"
ENABLE_LLM_HEADER_REPAIR = True  # if name still looks broken, ask LLM to recover header line
HF_MODEL_FOR_REPAIR = "meta-llama/Llama-3.1-70B-Instruct"  # uses the same token if present

# ----------------------------
# Low-level PDF/DOCX to text
# ----------------------------
def _normalize_ascii(text: str) -> str:
    # remove non-ascii artifacts
    return text.encode("ascii", "ignore").decode()

def _normalize_bullets(text: str) -> str:
    # turn any bullet-like glyphs into "\n• " at start of line
    text = re.sub(r"[•\u2022\*•\-]\s*", "\n• ", text)
    # collapse accidental double bullets
    text = re.sub(r"(?:\n•\s*){2,}", "\n• ", text)
    return text

def _collapse_spaces(text: str) -> str:
    # keep newlines (section boundaries), collapse intra-line whitespace
    text = re.sub(r"[ \t]{2,}", " ", text)
    # strip trailing spaces
    text = "\n".join(ln.rstrip() for ln in text.splitlines())
    return text.strip()

def _normalize_text(text: str) -> str:
    return _collapse_spaces(_normalize_bullets(_normalize_ascii(text)))

def parse_pdf_clean(path: str) -> str:
    # 1) native text
    try:
        with pdfplumber.open(path) as pdf:
            txt = "\n".join([p.extract_text() or "" for p in pdf.pages])
            if len((txt or "").strip()) > 20:
                return _normalize_text(txt)
    except Exception:
        pass
    # 2) PyPDF2 fallback
    try:
        with open(path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            txt = "\n".join([(pg.extract_text() or "") for pg in reader.pages])
            if len((txt or "").strip()) > 20:
                return _normalize_text(txt)
    except Exception:
        pass
    # 3) OCR fallback
    ocr_chunks = []
    try:
        with pdfplumber.open(path) as pdf:
            for p in pdf.pages:
                try:
                    img = p.to_image(resolution=300).original
                    ocr_chunks.append(pytesseract.image_to_string(img))
                except Exception:
                    continue
    except Exception:
        pass
    return _normalize_text("\n".join(ocr_chunks))

def parse_docx_clean(path: str) -> str:
    d = docx.Document(path)
    lines: List[str] = []
    for p in d.paragraphs:
        t = (p.text or "").strip()
        if not t:
            continue
        # treat list styles as bullets
        try:
            if p.style and p.style.name and p.style.name.lower().startswith("list"):
                t = "• " + t
        except Exception:
            # some docx have no style object
            pass
        lines.append(t)
    return _normalize_text("\n".join(lines))

def parse_resume_text_raw(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return parse_pdf_clean(path)
    elif ext == ".docx":
        return parse_docx_clean(path)
    else:
        raise ValueError(" Only PDF or DOCX supported.")

# --------------------------------
# TEI-ish entity & header repairs
# --------------------------------
_LINK_RX = re.compile(r"(https?://)?(www\.)?(linkedin\.com/[A-Za-z0-9_/\-]+|github\.com/[A-Za-z0-9_\-./]+)", re.I)
_PARTIAL_LI_RX = re.compile(r"\blinkedin/([A-Za-z0-9_\-\/]+)\b", re.I)
_EMAIL_RX = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
_FULL_PHONE_DIGITS = re.compile(r"(\+?1)?\D*(\d{3})\D*(\d{3})\D*(\d{4})")

def _repair_broken_caps_name(line: str) -> str:
    """
    Join spaced capital letters
    Applies only to the first line heuristically.
    """
    tokens = line.strip().split()
    # if many one-letter tokens, join them with neighbors
    if sum(1 for t in tokens if len(t) == 1 and t.isalpha()) >= 2:
        joined = "".join(tokens)
        rebuilt = []
        buff = []
        for t in tokens:
            if len(t) == 1 and t.isalpha():
                buff.append(t)
            else:
                if buff:
                    rebuilt.append("".join(buff))
                    buff = []
                rebuilt.append(t)
        if buff:
            rebuilt.append("".join(buff))
        line = " ".join(rebuilt)
    return line

def _extract_links(text: str) -> List[str]:
    links = set()
    for m in _LINK_RX.finditer(text):
        url = m.group(0)
        if not url.lower().startswith("http"):
            url = "https://" + url
        links.add(url.rstrip(").,;"))

    for m in _PARTIAL_LI_RX.finditer(text):
        slug = m.group(1).lstrip("/ ")
        links.add(f"https://linkedin.com/in/{slug}")
    return list(links)

def _extract_email(text: str) -> Optional[str]:
    m = _EMAIL_RX.search(text)
    return m.group(0) if m else None

def _extract_phone_ats(text: str) -> Optional[str]:
    # prefer first full 10-digit match anywhere in the document
    m = _FULL_PHONE_DIGITS.search(text)
    if not m:
        return None
    _, a, b, c = m.groups()
    return f"{a}-{b}-{c}" if PHONE_OUTPUT_FORMAT == "ats" else f"({a}) {b}-{c}"

def _inject_summary_if_missing(text: str) -> str:
    """
    If no SUMMARY/OBJECTIVE headings exist, infer a summary block between
    top header and the first major section (EDUCATION/EXPERIENCE/SKILLS/PROJECTS/CERTIFICATIONS).
    We add a 'SUMMARY' heading so Block-4 can segment it reliably.
    """
    low = text.lower()
    if any(h in low for h in ("summary\n", "\nsummary", "objective\n", "\nobjective", "\nprofile", "profile\n")):
        return text  # already there

    # find first big section anchor
    anchors = ["\neducation", "\nexperience", "\nwork experience", "\nskills", "\nprojects", "\ncertifications"]
    first_anchor_pos = min([low.find(a) for a in anchors if low.find(a) != -1] or [len(text)])

    # take the block after the top line up to the first anchor
    lines = text.splitlines()
    if not lines:
        return text
    # assume first 1–2 lines are name/contact. Summary begins from line 2 onward
    candidate = "\n".join(lines[1:])[:first_anchor_pos]
    # pick first 2–4 non-empty sentences/lines as summary
    cand_lines = [ln.strip() for ln in candidate.splitlines() if ln.strip() and not ln.strip().startswith("•")]
    cand_lines = [ln for ln in cand_lines if len(ln) > 20]  # avoid very short fragments
    if len(cand_lines) >= 1:
        snippet = "\n".join(cand_lines[:3])
        # inject SUMMARY heading before snippet
        return lines[0] + "\nSUMMARY\n" + snippet + "\n" + "\n".join(lines[1:])
    return text

def _llm_header_repair_if_needed(text: str) -> str:
    """
    If the first line looks like a likely broken name, ask the LLM
    to minimally repair only the first two lines: NAME and CONTACT.
    """
    try:
        lines = [ln for ln in text.splitlines() if ln.strip()]
        if not lines:
            return text

        first = lines[0].strip()

        # Suspicious-name heuristics
        tokens = first.split()
        suspicious = False

        # Too short overall
        if len(first) < 8:
            suspicious = True

        # Looks like proper name but many words missing first capital letters
        if len(tokens) >= 2:
            bad_titlecase = sum(
                1 for t in tokens
                if len(t) > 1 and t[0].islower()
            )
            if bad_titlecase > 0:
                suspicious = True

        # Example: Ekhana Evi Kuti, where all tokens are title-case
        # but likely missing initials from a known 3-part name pattern
        if len(tokens) >= 3:
            if all(t[:1].isupper() for t in tokens if t):
                # if multiple tokens are unusually short-ish, could be truncated
                shortish = sum(1 for t in tokens if len(t) <= 4)
                if shortish >= 2:
                    suspicious = True

        if not suspicious or not ENABLE_LLM_HEADER_REPAIR:
            return text

        client = InferenceClient(
            model=HF_MODEL_FOR_REPAIR,
            api_key=os.environ.get("HF_TOKEN"),
            timeout=60
        )

        header_scope = "\n".join(text.splitlines()[:6])
        prompt = f"""
You will receive the top header of a resume. Repair only obvious extraction issues.

Return EXACTLY TWO lines:
1) Full candidate name
2) One contact line

Do not invent details. Only fix obvious broken spacing or dropped leading letters if strongly implied.

Header:
{header_scope}
""".strip()

        resp = client.chat.completions.create(
            model=HF_MODEL_FOR_REPAIR,
            messages=[
                {"role": "system", "content": "You fix resume header extraction errors conservatively."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=120,
        )

        repaired = [ln.strip() for ln in resp.choices[0].message.content.strip().splitlines() if ln.strip()]
        if len(repaired) >= 2:
            raw_lines = text.splitlines()
            nonempty = [i for i, ln in enumerate(raw_lines) if ln.strip()]
            if nonempty:
                raw_lines[nonempty[0]] = repaired[0]
                if len(nonempty) > 1:
                    raw_lines[nonempty[1]] = repaired[1]
                else:
                    raw_lines.insert(nonempty[0] + 1, repaired[1])
            return "\n".join(raw_lines)

        return text
    except Exception:
        return text

def _repair_header_and_entities(text: str, is_pdf: bool) -> str:
    """
    Pipeline of small, safe repairs:
      - fix broken CAPS name on line 1
      - enrich links (complete linkedin/github)
      - recover phone (ATS) if found
      - inject SUMMARY heading if missing (helps Block-4)
      - optional: LLM header repair if still broken
    """
    lines = text.splitlines()
    if lines:
        lines[0] = _repair_broken_caps_name(lines[0])

    # enrich links (used later by Block-4 CONTACT slice)
    links = _extract_links(text)
    email = _extract_email(text)
    phone = _extract_phone_ats(text)

    # Rebuild contact line 2 if it's mostly fragments (helps segmentation)
    # Keep original but append canonical pieces we found.
    if len(lines) > 1:
        frag = lines[1].strip()
        parts = [frag] if frag else []
        if phone and phone not in frag: parts.append(phone)
        if email and email not in frag: parts.append(email)
        for u in links:
            if u not in frag:
                parts.append(u)
        lines[1] = " | ".join([p for p in parts if p])

    text2 = "\n".join(lines)
    text2 = _inject_summary_if_missing(text2)

    # Optional: if first line still looks broken, ask LLM to patch header minimally
    text2 = _llm_header_repair_if_needed(text2)

    return text2
def _looks_like_name(line: str) -> bool:
    toks = re.findall(r"[A-Za-z]+", line)
    if not (2 <= len(toks) <= 4):
        return False
    good = sum(1 for t in toks if t[:1].isupper())
    return good >= 2

def _ocr_pdf_header(path: str) -> str:
    """
    OCR only the top part of the first PDF page.
    This is usually where name + contact live.
    """
    try:
        import pdfplumber
        from PIL import Image

        with pdfplumber.open(path) as pdf:
            if not pdf.pages:
                return ""

            page = pdf.pages[0]
            page_width = page.width
            page_height = page.height

            # top 22% of first page
            crop_box = (0, 0, page_width, page_height * 0.22)
            cropped = page.crop(crop_box)

            img = cropped.to_image(resolution=300).original
            text = pytesseract.image_to_string(img)
            return _normalize_text(text)
    except Exception as e:
        print(f"⚠️ Header OCR failed: {e}")
        return ""

def _repair_header_from_ocr(path: str, parsed_text: str) -> str:
    """
    If OCR gives a better-looking header line than native extraction,
    replace the first non-empty line with the OCR name line.
    """
    try:
        ocr_header = _ocr_pdf_header(path)
        if not ocr_header.strip():
            return parsed_text

        parsed_lines = parsed_text.splitlines()
        parsed_nonempty = [i for i, ln in enumerate(parsed_lines) if ln.strip()]
        if not parsed_nonempty:
            return parsed_text

        parsed_name = parsed_lines[parsed_nonempty[0]].strip()

        ocr_lines = [ln.strip() for ln in ocr_header.splitlines() if ln.strip()]
        if not ocr_lines:
            return parsed_text

        ocr_name = ocr_lines[0]

        # use OCR name if it looks more like a proper name than parsed one
        parsed_tokens = re.findall(r"[A-Za-z]+", parsed_name)
        ocr_tokens = re.findall(r"[A-Za-z]+", ocr_name)

        parsed_score = sum(len(t) for t in parsed_tokens)
        ocr_score = sum(len(t) for t in ocr_tokens)

        if _looks_like_name(ocr_name) and ocr_score > parsed_score:
            parsed_lines[parsed_nonempty[0]] = ocr_name

            # also replace second line with OCR second line if available and useful
            if len(ocr_lines) > 1 and len(parsed_nonempty) > 1:
                parsed_lines[parsed_nonempty[1]] = ocr_lines[1]

            return "\n".join(parsed_lines)

        return parsed_text
    except Exception as e:
        print(f"⚠️ OCR header repair failed: {e}")
        return parsed_text
def parse_resume_text(path: str) -> str:
    """
    Final Layer 1 entry point.
    Produces robust text for structuring.
    """
    raw = parse_resume_text_raw(path)
    is_pdf = path.lower().endswith(".pdf")

    repaired = _repair_header_and_entities(raw, is_pdf=is_pdf)

    # extra OCR-based repair only for PDFs
    if is_pdf:
        repaired = _repair_header_from_ocr(path, repaired)

    return repaired

In [ ]:
import os

os.environ["HF_TOKEN"] = "yours"
print("✅ Hugging Face token loaded")

# **4. Section-wise Resume Structuring**

Once the text is cleaned, I move into section segmentation and structured extraction.

Here I define richer Pydantic models:

Contact
EducationItem
SkillGroup
ExperienceItem
ProjectItem
CertificationItem
and the full StructuredResume

This is more detailed than the earlier flat UserProfile, because by this point I want the resume represented in a way that later layers can reason over properly.

Then I build a prompt for the LLM that gives it coarse sections such as:

name
contact
summary
education
experience
projects
skills
certifications

The model’s job here is not to rewrite anything. It only structures what is already present into valid JSON.

I explicitly tell it to:

preserve bullet wording
not invent facts
separate bullets cleanly
infer dates only when clearly implied
and return a strict JSON object

I also made the date handling richer by storing both human-readable fields like start_date and end_date, and a normalized period field for consistency.

This is useful later, because I may want human-friendly display for export and structured date spans for reasoning.

So this part turns raw resume text into a structured resume object that the rest of the architecture can use.

# **5. Fallback Safety**

I also built a fallback in case the LLM structuring step fails.

If the model does not return valid JSON, the system falls back to a simpler heuristic-based structuring approach.

That means the extraction pipeline does not completely break just because one model call fails.

This was important to me because I wanted Layer 1 to be robust, not dependent on one perfect model response.

In [ ]:
# section-wise LLM Structuring + Inference (Dates = Option C) ===
import os, json, re, time, datetime
from typing import List, Dict, Optional, Union
from pydantic import BaseModel, Field
from huggingface_hub import InferenceClient

# ---------------------------------------------------------------------
# Fix deprecation for created_at elsewhere in your models
# created_at: str = Field(default_factory=lambda: datetime.datetime.now(datetime.UTC).isoformat())
# ---------------------------------------------------------------------

LLAMA_MODEL = "meta-llama/Llama-3.1-70B-Instruct"
_llama = InferenceClient(model=LLAMA_MODEL, api_key=os.environ.get("HF_TOKEN"), timeout=180)

# ---------- Structured schema (kept rich for research) ----------
class Contact(BaseModel):
    phone: Optional[str] = None
    email: Optional[str] = None
    location: Optional[str] = None
    links: List[str] = Field(default_factory=list)

class EducationItem(BaseModel):
    institution: Optional[str] = None
    degree: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = None     # human ("Aug 2021") or ISO ("2021-08")
    end_date: Optional[str] = None       # human ("May 2025") or "Present"
    period: Optional[str] = None         # ISO span e.g. "2021-08 – 2025-05" or "2021-08 – Present"
    gpa: Optional[str] = None
    coursework: List[str] = Field(default_factory=list)
    bullets: List[str] = Field(default_factory=list)

class SkillGroup(BaseModel):
    category: Optional[str] = None
    items: List[str] = Field(default_factory=list)

class ExperienceItem(BaseModel):
    title: Optional[str] = None
    organization: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    period: Optional[str] = None         # ISO span string, e.g. "2025-05 – Present"
    bullets: List[str] = Field(default_factory=list)

class ProjectItem(BaseModel):
    title: Optional[str] = None
    tech: List[str] = Field(default_factory=list)
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    period: Optional[str] = None
    bullets: List[str] = Field(default_factory=list)

class CertificationItem(BaseModel):
    title: Optional[str] = None
    issuer: Optional[str] = None
    year: Optional[str] = None

class StructuredResume(BaseModel):
    name: Optional[str] = None
    contact: Contact = Field(default_factory=Contact)
    summary: List[str] = Field(default_factory=list)
    education: List[EducationItem] = Field(default_factory=list)
    skills: List[SkillGroup] = Field(default_factory=list)
    experience: List[ExperienceItem] = Field(default_factory=list)
    projects: List[ProjectItem] = Field(default_factory=list)
    certifications: List[CertificationItem] = Field(default_factory=list)

# ---------- Helpers ----------
_JSON_BLOCK = re.compile(r"\{[\s\S]*\}")

def _json_only(text: str) -> Optional[dict]:
    m = _JSON_BLOCK.search(text or "")
    if not m: return None
    try:
        return json.loads(m.group())
    except Exception:
        return None

def _safe_section(s: Optional[str]) -> str:
    return (s or "").strip()

def _build_structuring_prompt(sectioned: Dict[str, str]) -> str:
    """
    sectioned keys expected:
      name, contact, summary, education, experience, projects, skills, certifications
    Values are raw text blocks from Block-3 (already segmented).
    We ask only for structuring + inference (no rewriting).
    """
    return f"""
You are a precise structuring engine. Convert the provided resume SECTIONS into a SINGLE JSON object
with the EXACT schema below. Do NOT invent facts. Preserve wording in bullets. Infer title/org/dates
only when clearly implied by the text. Dates MUST follow Option C: include human-friendly "start_date"/"end_date"
AND the ISO-span "period" (e.g., "2021-08 – 2025-05" or "2025-05 – Present").

Return JSON ONLY, with this schema:

{{
  "name": "string or null",
  "contact": {{
    "phone": "string or null",
    "email": "string or null",
    "location": "string or null",
    "links": ["..."]
  }},
  "summary": ["..."],

  "education": [
    {{
      "institution": "string or null",
      "degree": "string or null",
      "location": "string or null",
      "start_date": "string or null",
      "end_date": "string or null",
      "period": "string or null",
      "gpa": "string or null",
      "coursework": ["..."],
      "bullets": ["..."]
    }}
  ],

  "skills": [
    {{
      "category": "string or null",
      "items": ["..."]
    }}
  ],

  "experience": [
    {{
      "title": "string or null",
      "organization": "string or null",
      "location": "string or null",
      "start_date": "string or null",
      "end_date": "string or null",
      "period": "string or null",
      "bullets": ["..."]
    }}
  ],

  "projects": [
    {{
      "title": "string or null",
      "tech": ["..."],
      "start_date": "string or null",
      "end_date": "string or null",
      "period": "string or null",
      "bullets": ["..."]
    }}
  ],

  "certifications": [
    {{
      "title": "string or null",
      "issuer": "string or null",
      "year": "string or null"
    }}
  ]
}}

Rules:
- Do not summarize or paraphrase; copy bullet text as-is.
- Keep each bullet as a separate list item.
- If a field is missing, use null or empty list as applicable.
- For dates, include both "start_date"/"end_date" (human-readable) AND "period" (ISO span).
- Identify links (LinkedIn/GitHub/etc.) from the CONTACT block.

--- SECTIONS START ---
# NAME
{_safe_section(sectioned.get("name"))}

# CONTACT
{_safe_section(sectioned.get("contact"))}

# SUMMARY
{_safe_section(sectioned.get("summary"))}

# EDUCATION
{_safe_section(sectioned.get("education"))}

# EXPERIENCE
{_safe_section(sectioned.get("experience"))}

# PROJECTS
{_safe_section(sectioned.get("projects"))}

# SKILLS
{_safe_section(sectioned.get("skills"))}

# CERTIFICATIONS
{_safe_section(sectioned.get("certifications"))}
--- SECTIONS END ---
""".strip()

def _build_sections_from_parsed_text(parsed_text: str) -> Dict[str, str]:
    """
    Take the Block-3 'raw' (preferred) or 'clean' text and create coarse sections by headings.
    Your parse already prints perfect headings; this simply slices by anchors.
    """
    # Use your existing segmenter if you like. Here we do a narrow, robust split.
    text = parsed_text or ""
    lines = [ln.rstrip() for ln in text.splitlines()]

    # Normalize common headings
    anchors = {
        "summary": ["summary", "objective", "profile"],
        "education": ["education", "academics", "coursework"],
        "experience": ["work experience", "experience", "employment", "work history"],
        "skills": ["skills", "technical skills", "tech stack", "technologies", "tools"],
        "projects": ["projects", "academic projects", "course projects", "personal projects"],
        "certifications": ["certifications", "certification", "licenses", "licenses & certifications"],
    }

    # Identify name & contact (first 3–5 lines)
    top = "\n".join(lines[:6])
    # Hacky split: name line is the very first non-empty
    first_nonempty = next((ln for ln in lines if ln.strip()), "")
    name_block = first_nonempty.strip()
    # Contact: rest of top minus name (emails/phones/links)
    contact_block = "\n".join([ln for ln in lines[1:6] if ln.strip()])

    # Build sections by scanning headings
    sec_map = {"name": name_block, "contact": contact_block}
    current_key = None
    buf = []
    def flush():
        nonlocal current_key, buf
        if current_key:
            sec_map[current_key] = "\n".join(buf).strip()
        buf = []

    def as_key(h):
        h_low = h.lower().strip(": ")
        for k, alias in anchors.items():
            if any(h_low == a for a in alias):
                return k
        return None

    for ln in lines:
        k = as_key(ln)
        if k:
            flush()
            current_key = k
            buf = []
            continue
        if current_key:
            buf.append(ln)
    flush()

    # Ensure all exist
    for k in ["summary","education","experience","projects","skills","certifications"]:
        sec_map.setdefault(k, "")

    return sec_map

# ---------- Main: structure with Llama ----------
def struct_from_sections(parsed_text: str) -> Dict:
    prompt = _build_structuring_prompt(_build_sections_from_parsed_text(parsed_text))
    tries = 2
    last = None
    for _ in range(tries):
        try:
            resp = _llama.chat.completions.create(
                model=LLAMA_MODEL,
                messages=[
                    {"role":"system","content":"Return JSON only. No commentary."},
                    {"role":"user","content": prompt}
                ],
                max_tokens=4000,
                temperature=0.0,
            )
            content = resp.choices[0].message.content
            obj = _json_only(content)
            if not obj:
                raise ValueError("No JSON detected in model output.")
            return StructuredResume(**obj).model_dump()
        except Exception as e:
            last = e
            time.sleep(1.0)
    # If LLM fails, fall back to your heuristic (still returns dict)
    print(f"⚠️ Structuring LLM failed, fallback used: {last}")
    secs = _build_sections_from_parsed_text(parsed_text)
    # Minimal heuristic fallback (flat)
    return StructuredResume(
        name=secs.get("name") or None,
        contact=Contact(),
        summary=[ln for ln in secs.get("summary","").splitlines() if ln.strip()],
        education=[EducationItem(bullets=[ln for ln in secs.get("education","").splitlines() if ln.strip()])],
        skills=[SkillGroup(items=[ln for ln in secs.get("skills","").splitlines() if ln.strip()])],
        experience=[ExperienceItem(bullets=[ln for ln in secs.get("experience","").splitlines() if ln.strip()])],
        projects=[ProjectItem(bullets=[ln for ln in secs.get("projects","").splitlines() if ln.strip()])],
        certifications=[CertificationItem(title=ln) for ln in secs.get("certifications","").splitlines() if ln.strip()],
    ).model_dump()

# ---------- Profile glue (keep your schema untouched) ----------
def _pretty_edu(e: EducationItem) -> str:
    parts = [p for p in [e.institution, e.degree] if p]
    tail = f" — {e.period}" if e.period else ""
    return (" | ".join(parts) + tail) if parts else (e.period or "")

def _pretty_exp(x: ExperienceItem) -> List[str]:
    header = " | ".join([p for p in [x.title, x.organization, x.location] if p])
    date = f" — {x.period}" if x.period else ""
    header_line = (header + date).strip(" |—")
    return [header_line] + (x.bullets or []) if header_line else (x.bullets or [])

def normalize_resume_with_structuring(parsed_text: str, profile: Optional[UserProfile] = None) -> UserProfile:
    prof = profile or UserProfile()
    data = struct_from_sections(parsed_text)

    # Save the full rich JSON for research / later layers
    sid = prof.session_id
    ensure_dir_exists(f"data/{sid}.extracted.json")
    with open(f"data/{sid}.extracted.json","w") as f:
        json.dump(data, f, indent=2)

    # Map a readable subset into your existing UserProfile fields
    sr = StructuredResume(**data)

    # Topline
    if sr.name: prof.name = sr.name
    if sr.contact.email: prof.email = sr.contact.email
    if sr.contact.location: prof.location = sr.contact.location
    if sr.contact.phone: prof.phone = sr.contact.phone
    if sr.contact.links:
        # opportunistic linkedin
        for u in sr.contact.links:
            if "linkedin.com" in u.lower():
                prof.linkedin = u; break
        prof.any_links = sr.contact.links

    # Lists (display-friendly while we keep rich JSON on disk)
    prof.summary = sr.summary

    prof.education = [_pretty_edu(e) for e in sr.education] if sr.education else []
    # Experience: include header + bullets flattened as lines (for quick UI), rich lives in extracted.json
    flat_exp: List[str] = []
    for x in sr.experience:
        flat_exp += _pretty_exp(x)
    prof.experience = flat_exp

    # Skills: show as "Category: item1, item2"
    prof.skills = [ (f"{g.category}: " if g.category else "") + ", ".join(g.items) for g in sr.skills ] if sr.skills else []

    # Projects: show "Title — period" then bullets
    flat_proj: List[str] = []
    for p in sr.projects:
        head = p.title or ""
        if p.period: head = f"{head} — {p.period}" if head else p.period
        if head: flat_proj.append(head)
        flat_proj += (p.bullets or [])
    prof.projects = flat_proj

    # Certifications
    prof.certifications = [ " | ".join([x for x in [c.title, c.issuer, c.year] if x]) for c in sr.certifications ] if sr.certifications else []

    save_json(prof, profile_path(prof.session_id))
    print(f"✅ Structured extraction saved → data/{sid}.extracted.json")
    return prof

print("✅ Section-wise Structuring, Date Format C loaded.")

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient("meta-llama/Llama-3.1-70B-Instruct", token=os.environ["HF_TOKEN"])
print("✅ Connected to:", client.model)

# **6. Normalization into the Working Profile**

Finally, after getting the full structured resume, I map a readable subset of it back into the simpler UserProfile.

This gives me:

a structured JSON file saved on disk for later layers
and a flat, display-friendly version for quick inspection

So I save two things:

a rich extracted resume file
and the simpler session profile file

This makes debugging easier and keeps the system flexible.

In [ ]:
# === RUN EXTRACTION ===
parsed_text = parse_resume_text("/content/RESUME2.pdf")

profile = normalize_resume_with_structuring(parsed_text)

print("✅ DONE — Extracted & structured profile:")
print(json.dumps(load_json(profile_path(profile.session_id)), indent=2))

In [ ]:
def detect_resume_template(parsed_text: str) -> Dict:
    lines = [l.rstrip() for l in parsed_text.splitlines()]
    template = {}

    # ----------------------------
    # Section heading style
    # ----------------------------
    headers = [l.strip() for l in lines if l.strip().isupper() and 1 <= len(l.split()) <= 4]
    template["section_heading_case"] = "uppercase" if headers else "normal"

    # detect section order
    section_candidates = [
        "SUMMARY",
        "EDUCATION",
        "SKILLS",
        "TECHNICAL SKILLS",
        "WORK EXPERIENCE",
        "EXPERIENCE",
        "PROJECTS",
        "CERTIFICATIONS"
    ]

    section_order = []
    for l in lines:
        clean = l.strip().upper()
        if clean in section_candidates and clean not in section_order:
            section_order.append(clean)

    template["section_order"] = section_order

    # ----------------------------
    # Bullet style
    # ----------------------------
    bullets = [l.strip()[0] for l in lines if l.strip().startswith(("•", "-", "*"))]
    template["bullet_style"] = bullets[0] if bullets else "•"

    # ----------------------------
    # Skills formatting
    # ----------------------------
    colon_lines = [l for l in lines if ":" in l and "," in l]
    template["skills_layout"] = "category_colon_list" if len(colon_lines) >= 2 else "flat_list"

    # ----------------------------
    # Project formatting
    # ----------------------------
    pipe_lines = [l for l in lines if "|" in l]
    template["project_layout"] = "pipe_format" if len(pipe_lines) >= 2 else "standard"

    # ----------------------------
    # Date format detection
    # ----------------------------
    month_year = re.search(
        r"\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4}\b",
        parsed_text,
        re.IGNORECASE
    )

    iso_like = re.search(r"\b\d{4}-\d{2}\b", parsed_text)

    if month_year:
        template["date_format"] = "human_month_year"
    elif iso_like:
        template["date_format"] = "iso"
    elif re.search(r"\b\d{4}\b", parsed_text):
        template["date_format"] = "year_only"
    else:
        template["date_format"] = "unknown"

    # ----------------------------
    # Vertical spacing density
    # ----------------------------
    blank_lines = sum(1 for l in lines if not l.strip())
    template["has_blank_line_spacing"] = blank_lines > 5

    # estimate spacing density
    avg_line_length = sum(len(l) for l in lines if l.strip()) / max(1, len([l for l in lines if l.strip()]))

    if avg_line_length > 80:
        template["density"] = "compact"
    elif avg_line_length > 60:
        template["density"] = "medium"
    else:
        template["density"] = "spacious"

    # ----------------------------
    # Header style detection
    # ----------------------------
    template["experience_layout"] = "two_line_header"
    template["education_layout"] = "two_line_header"

    # ----------------------------
    # Contact line separator
    # ----------------------------
    first_lines = " ".join(lines[:5])

    if "|" in first_lines:
        template["contact_separator"] = "|"
    elif "•" in first_lines:
        template["contact_separator"] = "•"
    else:
        template["contact_separator"] = "space"

    # ----------------------------
    # Detect one-page density hint
    # ----------------------------
    non_empty = [l for l in lines if l.strip()]
    blank = [l for l in lines if not l.strip()]
    template["estimated_line_count"] = len(non_empty)

    blank_ratio = len(blank) / max(1, len(lines))

    # compact resumes have very few blank lines
    if blank_ratio < 0.15 and len(non_empty) < 80:
        template["page_density"] = "one_page_compact"
    else:
        template["page_density"] = "multi_page"

    # refine density label
    avg_len = sum(len(l) for l in non_empty) / max(1, len(non_empty))

    if avg_len > 70:
        template["density"] = "compact"
    elif avg_len > 50:
        template["density"] = "medium"
    else:
        template["density"] = "spacious"


    return template

In [ ]:
template_profile = detect_resume_template(parsed_text)

with open(f"data/{profile.session_id}.template.json", "w") as f:
    json.dump(template_profile, f, indent=2)

In [ ]:
import json
print(json.dumps(
    json.load(open(f"data/{profile.session_id}.template.json")),
    indent=2
))

# **What Layer 1 Achieves**

By the end of Layer 1, I have taken a raw resume file and transformed it into a structured, validated, persistent resume representation.

That means later layers do not need to worry about:

file parsing
broken formatting
missing headings
messy bullets
or inconsistent contact extraction

They can just work with structured data.

So overall, Layer 1 is the foundation of the whole project. It handles resume intake, cleaning, repair, and structured extraction in a way that makes the rest of the pipeline much more reliable.

# **Layer 2: JD Intelligence (Dynamic, Model-Driven Version)**

In Layer 2, I built the job description intelligence pipeline. The goal of this layer is to take a raw job description and convert it into a structured feature map that the later layers can use for alignment, scoring, and optimization.

If Layer 1 is about understanding the resume, then Layer 2 is about understanding the job description in a structured way.

I did not want to rely on a fixed list of hardcoded skills or manually defined keywords, because that would make the system rigid and weak when dealing with different kinds of roles. Instead, I wanted the system to dynamically discover what matters in a given job description.

So Layer 2 is designed to do four main things:

extract important terms from the JD
organize them into meaningful categories
remove semantic duplicates
and assign weights so the later alignment stage knows what matters most

# **1. Loading the Core NLP and LLM Tools**

The first step in this layer is loading the models and libraries needed for JD understanding.

I use spaCy for linguistic structure, especially noun phrase extraction.

I use KeyBERT for keyword extraction, because I wanted something lightweight but context-aware instead of only relying on frequency-based terms.

I use Sentence Transformers, specifically all-MiniLM-L6-v2, for semantic encoding and similarity.

And I use the Llama model through Hugging Face for higher-level reasoning, mainly for categorizing the extracted terms into semantic buckets.

So this layer combines classical NLP, keyword extraction, embeddings, and LLM reasoning in one workflow.

That was intentional. I did not want the system to depend on only one method, because job descriptions vary too much in writing style and detail.

In [ ]:
# Core NLP and ML dependencies
!pip install -q spacy keybert sentence-transformers huggingface-hub

# **2. Cleaning the Job Description Text**

The clean_text function is simple but necessary.

It normalizes whitespace and strips extra spacing so that the later extraction steps are more stable.

This may seem small, but job descriptions copied from websites often come with weird spacing, broken lines, or inconsistent formatting. I wanted the pipeline to start from a clean version before doing any NLP.

So this part is basic preprocessing.

# **3. Dynamic Term Extraction**

The next major step is extract_dynamic_terms.

This is where I begin discovering what the job description is actually about.

Instead of using a predefined skill vocabulary, I combine two signals:

spaCy noun phrases
and KeyBERT extracted keywords

The noun phrases help capture meaningful chunks like job concepts, tools, or responsibilities.

KeyBERT helps identify terms that are semantically central to the JD even if they do not appear as formal noun phrases.

Then I merge the two outputs and filter them.

I remove things that are too generic, too short, purely numeric, or likely to be noise. For example, words like “experience,” “project,” or “candidate” do not tell me much about the actual role requirements, so I filter those out.

This gives me a dynamic set of candidate terms that are specific to the current JD rather than coming from some fixed skill dictionary.

That was one of the important design choices in Layer 2. I wanted the extraction to be adaptive and domain-independent.

# **4. LLM-Based Categorization**

Once I have the candidate terms, I pass them into categorize_with_llm.

This is where the LLM helps organize the extracted keywords into categories:

Hard_Skills
Soft_Skills
Tools
Certifications
Domain
Other

I included a short slice of the job description in the prompt so the model has context while assigning categories. That way it does not classify keywords in isolation.

The reason I used an LLM here instead of only rules is that many JD terms are ambiguous. A word may act like a tool in one job description and like a broader domain concept in another. The LLM helps resolve that ambiguity more intelligently than a purely hardcoded rule system would.

I also enforce JSON-only output, because I wanted the result to be machine-readable and easy to pass into later layers.

And if the LLM fails, I have a fallback that places all keywords into Hard_Skills so the pipeline does not collapse completely.

So this part gives the job description semantic structure.

# **5. Semantic Deduplication**

After categorization, I run deduplicate_semantic.

This part uses the sentence transformer model again, but now for term-to-term similarity.

The reason I added this is that keyword extraction often produces overlapping phrases. For example, a JD might produce several expressions that all mean nearly the same thing, like slightly different tool names, phrases, or repeated concepts.

I did not want those duplicates to inflate the feature map and make the later alignment stage noisy.

So I embed all extracted terms and compare them using cosine similarity. If two terms are too close semantically, I keep one and drop the duplicate.

This gives me a cleaner, more compact set of job description features.

So this stage is really about reducing redundancy before scoring.

# **6. Weight Computation**

Once the categories are cleaned, I compute weights for each keyword using compute_weights.

This is one of the most important parts of Layer 2.

For every extracted keyword, I compute its semantic similarity to the entire job description embedding. Then I normalize those scores so they fall into a comparable range.

The result is a weighted feature map.

That means Layer 3 will not just know what skills are present in the JD. It will also know which ones matter more.

This is useful because not every JD keyword should be treated equally.

Some terms are central to the role, while others are secondary or peripheral. By assigning normalized weights, the alignment stage can focus more strongly on the high-importance skills.

So this part transforms the JD from a list of terms into a prioritized semantic map.

In [ ]:
import os, re, json
from typing import Dict, List
from collections import defaultdict
import spacy
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer, util
from huggingface_hub import InferenceClient

# ---------- Load Models ----------
nlp = spacy.load("en_core_web_sm")
kw_model = KeyBERT(model="all-MiniLM-L6-v2")
sbert = SentenceTransformer("all-MiniLM-L6-v2")

LLAMA_MODEL = "meta-llama/Llama-3.1-70B-Instruct"
llm = InferenceClient(model=LLAMA_MODEL, api_key=os.environ.get("HF_TOKEN"), timeout=180)

# ---------- Helpers ----------
def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def extract_dynamic_terms(text: str, top_n: int = 50) -> List[str]:
    """Combine KeyBERT keywords and spaCy noun phrases, remove duplicates and generic phrases."""
    doc = nlp(text)
    noun_phrases = [chunk.text.strip() for chunk in doc.noun_chunks]
    keybert_terms = [kw for kw, _ in kw_model.extract_keywords(text, keyphrase_ngram_range=(1,3), top_n=top_n)]
    merged = list(set(noun_phrases + keybert_terms))

    # remove very short/generic or numeric terms
    stop_like = {"experience","project","task","role","responsibilities","requirement","candidate","position"}
    filtered = [
        t for t in merged
        if len(t.split()) <= 5
        and len(t) > 3
        and not any(w.lower() in stop_like for w in t.split())
        and not re.fullmatch(r"[\d\W]+", t)
    ]
    return filtered

def categorize_with_llm(keywords: List[str], jd_text: str) -> Dict[str, List[str]]:
    """Ask the LLM to categorize keywords into semantic buckets."""
    prompt = f"""
    Categorize each term into one of:
    - Hard_Skills
    - Soft_Skills
    - Tools
    - Certifications
    - Domain
    - Other

    Job Description:
    {jd_text[:1200]}

    Keywords:
    {keywords}

    Return ONLY valid JSON:
    {{
      "Hard_Skills": [],
      "Soft_Skills": [],
      "Tools": [],
      "Certifications": [],
      "Domain": [],
      "Other": []
    }}
    """
    try:
        resp = llm.chat.completions.create(
            model=LLAMA_MODEL,
            messages=[
                {"role":"system","content":"Return valid JSON only."},
                {"role":"user","content":prompt}
            ],
            max_tokens=1000,
            temperature=0.0
        )
        raw = resp.choices[0].message.content
        m = re.search(r"\{[\s\S]*\}", raw)
        return json.loads(m.group()) if m else {}
    except Exception as e:
        print(f"⚠️ LLM categorization failed: {e}")
        return {
            "Hard_Skills": keywords, "Soft_Skills": [],
            "Tools": [], "Certifications": [],
            "Domain": [], "Other": []
        }

def deduplicate_semantic(categories: Dict[str, List[str]]) -> Dict[str, List[str]]:
    """Deduplicate similar terms by cosine similarity."""
    all_terms = [(c, kw) for c, kws in categories.items() for kw in kws]
    if not all_terms: return categories

    terms = [kw for _, kw in all_terms]
    embs = sbert.encode(terms, convert_to_tensor=True)
    kept, used = [], []

    for i, kw in enumerate(terms):
        if any(util.cos_sim(embs[i], embs[j]) > 0.85 for j in used):
            continue
        used.append(i)
        kept.append(kw)

    # filter categories to only kept terms
    out = {c: [kw for kw in kws if kw in kept] for c, kws in categories.items()}
    return {k:v for k,v in out.items() if v}

def compute_weights(categories: Dict[str, List[str]], jd_text: str) -> Dict[str, Dict[str, float]]:
    jd_emb = sbert.encode(jd_text, convert_to_tensor=True)
    weights = {}
    all_scores = []
    raw = defaultdict(dict)
    for cat, kws in categories.items():
        for kw in kws:
            sc = float(util.cos_sim(sbert.encode(kw, convert_to_tensor=True), jd_emb))
            raw[cat][kw] = sc
            all_scores.append(sc)
    if not all_scores: return {}
    min_s, max_s = min(all_scores), max(all_scores)
    for cat, kw_map in raw.items():
        weights[cat] = {
            kw: round((sc - min_s)/(max_s - min_s + 1e-6),3)
            for kw, sc in kw_map.items() if sc > 0
        }
    return weights

# ---------- MAIN PIPELINE ----------
def build_jd_feature_map(jd_text: str, job_title: str=None) -> Dict:
    jd_text = clean_text(jd_text)
    terms = extract_dynamic_terms(jd_text)
    categories = categorize_with_llm(terms, jd_text)
    categories = deduplicate_semantic(categories)
    weights = compute_weights(categories, jd_text)
    cleaned = {k:list(v.keys()) for k,v in weights.items() if v}

    jd_map = {
        "job_title": job_title or "Untitled Role",
        "categories": cleaned,
        "weights": weights,
        "total_keywords": sum(len(v) for v in cleaned.values()),
        "vector_model": "all-MiniLM-L6-v2",
        "llm_model": LLAMA_MODEL
    }

    os.makedirs("data", exist_ok=True)
    with open("data/JD_Feature_Map.json", "w") as f:
        json.dump(jd_map, f, indent=2)

    print("✅ JD Feature Map saved → data/JD_Feature_Map.json")
    return jd_map

# **7. Building the JD Feature Map**

The final step is build_jd_feature_map.

This function ties together the whole pipeline:

clean the text
extract dynamic terms
categorize them
remove duplicates
compute weights
and package everything into a structured dictionary

The final output contains:

job title
categories
weights
total keyword count
the vector model used
and the LLM model used

Then I save it as data/JD_Feature_Map.json.

That saved file becomes the core input for Layer 3, where the resume is matched against the JD.

So by the end of Layer 2, the system has converted an unstructured job description into a structured, weighted semantic representation.

# **8. Interactive Testing**

I also added a simple testing interface where the user can enter a job title and paste the JD directly into the notebook.

That makes Layer 2 easy to test independently before integrating it with the rest of the pipeline.

It also helps me verify whether the extracted categories and weights look reasonable before moving into resume alignment.

So this testing block is mainly there for iterative validation and debugging.

In [ ]:
# === Single JD Test with User Input (Inline Version) ===
import json

# Make sure build_jd_feature_map() is already defined in your notebook before this cell.

job_title = input("Enter Job Title: ").strip()
print("\nPaste or type the full Job Description below (press Enter twice to finish):\n")

lines = []
while True:
    line = input()
    if not line:
        break
    lines.append(line)
jd_text = "\n".join(lines)

print("\n🔍 Extracting structured features, please wait...\n")
jd_map = build_jd_feature_map(jd_text, job_title=job_title)

print("✅ Done! Here’s your structured output:\n")
print(json.dumps(jd_map, indent=2))

# **What Layer 2 Achieves**

By the end of Layer 2, I no longer have just raw JD text.

I have a structured feature map that tells me:

what important concepts exist in the job description
how those concepts are categorized
which terms are duplicates and should be removed
and how strongly each term is related to the overall role

That structured JD representation is what makes Layer 3 possible.

Without this layer, the alignment stage would just be doing loose keyword matching. With Layer 2, it becomes a weighted semantic comparison problem.

So overall, Layer 2 is the intelligence layer for the job description. It transforms raw hiring language into something the system can reason over in a structured way.

# **LAYER 3**

Layer 3 is where the system moves from just understanding the resume and job description to actually optimizing the resume in a controlled way.

By this point, Layer 1 has already turned the resume into structured data, and Layer 2 has already turned the job description into a structured feature map. So Layer 3 takes those two inputs and does the real matching, reasoning, suggestion generation, selective editing, and final export preparation.

The goal of this layer is not to blindly rewrite the resume. The goal is to compare the resume against the job description, figure out what aligns well and what does not, generate targeted suggestions, let the user decide what to keep, and then assemble a polished final version.

So this layer acts like the main optimization engine of the project.

# **Block 1: Config, Data Loading, and Schemas**

The first block loads the two core inputs into Layer 3.

One is the structured resume JSON from Layer 1.
The other is the job description feature map from Layer 2.

I define the full schema again here using Pydantic so that Layer 3 can work with strongly structured objects instead of loose dictionaries. That includes contact information, education entries, skill groups, experience items, project items, certifications, and the JD feature map.

This block matters because it gives the rest of Layer 3 a clean interface.

Instead of saying “maybe this key exists,” the rest of the code can assume things like:

resume.skills is a list of skill groups
resume.experience is a list of experience entries
jd.categories contains grouped JD keywords
jd.weights contains importance scores

So Block 1 is basically the structured entry point into the optimization pipeline.

In [ ]:
# === LAYER 3 — Alignment & Resume Optimization Pipeline ===
# BLOCK 1: Config + Data Loading + Correct Schemas

import os
import json
from typing import List, Dict, Optional
from pydantic import BaseModel, Field

# ----- PATHS: update these for your run -----
# 1) Structured resume JSON from Layer 1 (upload OR chatbot)
RESUME_JSON_PATH = "/content/data/b7fe85d6-0fb4-4929-a0fc-fa45ac5b7cc3.extracted.json"

# 2) JD feature map JSON from Layer 2
JD_FEATURE_MAP_PATH = "/content/data/JD_Feature_Map.json"


# ----- Schemas (rich structured resume + JD map) -----
class Contact(BaseModel):
    phone: Optional[str] = None
    email: Optional[str] = None
    location: Optional[str] = None
    links: List[str] = Field(default_factory=list)

class EducationItem(BaseModel):
    institution: Optional[str] = None
    degree: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    period: Optional[str] = None
    gpa: Optional[str] = None
    coursework: List[str] = Field(default_factory=list)
    bullets: List[str] = Field(default_factory=list)

class SkillGroup(BaseModel):
    category: Optional[str] = None
    items: List[str] = Field(default_factory=list)

class ExperienceItem(BaseModel):
    title: Optional[str] = None
    organization: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    period: Optional[str] = None
    bullets: List[str] = Field(default_factory=list)

class ProjectItem(BaseModel):
    title: Optional[str] = None
    tech: List[str] = Field(default_factory=list)
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    period: Optional[str] = None
    bullets: List[str] = Field(default_factory=list)

class CertificationItem(BaseModel):
    title: Optional[str] = None
    issuer: Optional[str] = None
    year: Optional[str] = None
    link: Optional[str] = None

class StructuredResume(BaseModel):
    name: Optional[str] = None
    contact: Contact = Field(default_factory=Contact)
    summary: List[str] = Field(default_factory=list)
    education: List[EducationItem] = Field(default_factory=list)
    skills: List[SkillGroup] = Field(default_factory=list)
    experience: List[ExperienceItem] = Field(default_factory=list)
    projects: List[ProjectItem] = Field(default_factory=list)
    certifications: List[CertificationItem] = Field(default_factory=list)

class JDFeatureMap(BaseModel):
    job_title: Optional[str] = None
    categories: Dict[str, List[str]] = Field(default_factory=dict)
    weights: Dict[str, Dict[str, float]] = Field(default_factory=dict)
    total_keywords: Optional[int] = None
    # allow extra fields (like vector_model, llm_model) without error
    class Config:
        extra = "ignore"

# ----- Load helpers -----
def load_resume_struct(path: str) -> StructuredResume:
    with open(path, "r") as f:
        data = json.load(f)
    return StructuredResume(**data)

def load_jd_map(path: str) -> JDFeatureMap:
    with open(path, "r") as f:
        data = json.load(f)
    return JDFeatureMap(**data)

# ----- Test loading to verify schema correctness -----
resume_struct = load_resume_struct(RESUME_JSON_PATH)
jd_map = load_jd_map(JD_FEATURE_MAP_PATH)

print("✅ Loaded resume for:", resume_struct.name)
print("✅ Loaded JD feature map for role:", jd_map.job_title)
print("Resume — number of skill groups:", len(resume_struct.skills))
print("Resume — number of experience entries:", len(resume_struct.experience))
print("JD — total keywords:", jd_map.total_keywords)

# **Block 2: Semantic Alignment and Recommendation Buffer**

This block is the matching engine.

The idea here is to compare what is present in the resume against what the job description is asking for.

First, I flatten the relevant resume text into pools such as:

skills
summary
experience bullets
project bullets

Then, for each JD keyword, I compute semantic similarity between that keyword and all the resume text using sentence embeddings.

If the semantic similarity is strong enough, I count it as a match.

If semantic similarity is weak, I optionally fall back to a lexical overlap score. That gives the system another chance to detect simpler matches where the wording is close but semantic similarity alone may not be enough.

Then I store all of that in an alignment report.

That report includes:

overall alignment score
alignment score by category
detailed skill match records
recommendations for missing or weak but important skills

The recommendation logic is very important here.
If a skill has low match score but high JD weight, it gets flagged for improvement.

So Block 2 does two things at once:

it measures how well the resume aligns with the JD
and it creates the recommendation buffer that later blocks use for optimization

This is really the decision-support layer for the rest of Layer 3.

In [ ]:
# === LAYER 3 — Block 2 (improved): Semantic + Lexical Matching + Recommendation Buffer ===

from sentence_transformers import SentenceTransformer, util
import numpy as np
from typing import Any, Dict, List
import re

EMBED_MODEL = "all-MiniLM-L6-v2"
model = SentenceTransformer(EMBED_MODEL)

SEM_SIM_THRESHOLD = 0.45   # threshold for considering semantic similarity as “match”
LEXICAL_FALLBACK = True    # whether to fallback to lexical matching if semantic is low
LEXICAL_THRESHOLD = 0.6    # threshold for lexical matching (normalized overlap score)

def _flatten_resume_text(resume: StructuredResume) -> Dict[str, List[str]]:
    out = {"skills": [], "summary": [], "experience": [], "projects": []}
    for sg in resume.skills:
        out["skills"].extend(sg.items or [])
    out["summary"].extend(resume.summary or [])
    for e in resume.experience:
        out["experience"].extend(e.bullets or [])
    for p in resume.projects:
        out["projects"].extend(p.bullets or [])
    return out

def _lexical_score(a: str, b: str) -> float:
    """
    Simple lexical fallback: based on overlap of normalized tokens.
    Returns fraction of overlapping tokens between a and b.
    """
    toks_a = re.findall(r"\w+", a.lower())
    toks_b = re.findall(r"\w+", b.lower())
    if not toks_a or not toks_b:
        return 0.0
    set_a = set(toks_a)
    set_b = set(toks_b)
    overlap = set_a & set_b
    return len(overlap) / max(len(set_a), len(set_b))

def compute_alignment_graded(resume: StructuredResume, jd: JDFeatureMap) -> Dict[str, Any]:
    pools = _flatten_resume_text(resume)
    # build resume embeddings
    all_text = []
    for pool in pools.values():
        all_text.extend(pool)
    if all_text:
        emb_resume = model.encode(all_text, convert_to_tensor=True)
    else:
        emb_resume = None

    report = {
        "overall": 0.0,
        "by_category": {},
        "skill_matches": [],      # list of {skill, category, score, how_matched}
        "recommendations": []
    }

    category_scores = {}
    for cat, skills in jd.categories.items():
        weighted_scores = []
        total_weight = 0.0
        for sk in skills:
            weight = jd.weights.get(cat, {}).get(sk, 0.0)
            score = 0.0
            method = None

            # Semantic match first
            emb_sk = model.encode(sk, convert_to_tensor=True)
            if emb_resume is not None:
                sims = util.cos_sim(emb_sk, emb_resume)[0].cpu().numpy()
                best = float(sims.max()) if sims.size > 0 else 0.0
            else:
                best = 0.0

            if best >= SEM_SIM_THRESHOLD:
                score = best
                method = "semantic"
            elif LEXICAL_FALLBACK:
                # Lexical fallback across all resume text
                best_lex = 0.0
                for pool in pools.values():
                    for txt in pool:
                        lex = _lexical_score(sk, txt)
                        if lex > best_lex:
                            best_lex = lex
                if best_lex >= LEXICAL_THRESHOLD:
                    score = best_lex * 0.75  # discount lexical matches a bit
                    method = "lexical"

            # record result
            weighted_scores.append((weight, score))
            total_weight += weight
            report["skill_matches"].append({
                "skill": sk,
                "category": cat,
                "jd_weight": weight,
                "match_score": score,
                "method": method or "none"
            })

            # if match is weak or missing but jd weight high → recommend
            if score < 0.5 and weight >= 0.3:
                report["recommendations"].append(sk)

        # aggregate category score as weighted average
        if total_weight > 0:
            cat_score = sum(w * s for w, s in weighted_scores) / total_weight
        else:
            cat_score = 0.0
        report["by_category"][cat] = cat_score

    if report["by_category"]:
        report["overall"] = float(np.mean(list(report["by_category"].values())))

    return report

# --- Run and display the improved alignment + recommendation buffer ---
alignment2 = compute_alignment_graded(resume_struct, jd_map)

import json
print("=== IMPROVED ALIGNMENT REPORT ===")
print("Overall score:", alignment2["overall"])
print("By category:", alignment2["by_category"])
print("Missing or weak but important skills (recommendations):", alignment2["recommendations"])
print("\nSample skill match details (first 10):")
print(json.dumps(alignment2["skill_matches"][:10], indent=2))

# **Block 3: ATS Keyword Optimization and Phrasing Suggestions**

Once Block 2 tells us what is strong, partial, and weak, Block 3 uses that information to generate targeted phrasing suggestions.

The first step is to organize alignment results into three buckets:

strong matches
partial matches
weak or missing matches

Then I select the top keywords from those buckets and build a prompt for the LLM.

But the prompt is very controlled.

The model is not told to rewrite the whole resume.
Instead, it is asked to suggest:

summary phrasing ideas
experience bullet phrasing ideas
project bullet phrasing ideas
keyword synonyms or ATS-friendly wording variants

It is also explicitly told:

do not invent facts
do not fabricate experience
prefer rewording over adding new content
keep suggestions subtle and human

That is an important part of the architecture.
This block is not performing final edits. It is generating a suggestion bundle that later blocks can apply selectively.

I also added caching here. That means if the same resume, JD, and alignment report are used again, the system reuses the saved optimization bundle instead of calling the LLM again. That makes repeated runs more stable and efficient.

So Block 3 acts like the controlled idea-generation stage of Layer 3.

In [ ]:
# === LAYER 3 — BLOCK 3: ATS Keyword Optimization + Phrasing Suggestions ===
# Depends on:
#   - resume_struct  (StructuredResume from Block 1)
#   - jd_map         (JDFeatureMap from Block 1)
#   - alignment2     (dict from Block 2: compute_alignment_graded)

import json
from typing import Any, Dict, List
from huggingface_hub import InferenceClient

LLM_MODEL = "meta-llama/Llama-3.1-70B-Instruct"
hf_client = InferenceClient(
    model=LLM_MODEL,
    api_key=os.environ.get("HF_TOKEN"),
    timeout=120,
)

def build_gap_summary(alignment_report: Dict[str, Any]) -> Dict[str, List[Dict[str, Any]]]:
    """
    Turn alignment2 into a clean gap structure we can both:
      - inspect in Python
      - feed into LLM prompts.
    """
    strong, partial, weak_or_missing = [], [], []
    for m in alignment_report["skill_matches"]:
        w = m["jd_weight"]
        s = m["match_score"]
        rec = {
            "skill": m["skill"],
            "category": m["category"],
            "weight": w,
            "score": s,
            "method": m["method"],
        }
        if s >= 0.70:
            strong.append(rec)
        elif 0.40 <= s < 0.70:
            partial.append(rec)
        else:
            # treat everything < 0.40 as weak / missing
            weak_or_missing.append(rec)

    # Sort inside each bucket by JD weight descending
    strong.sort(key=lambda x: x["weight"], reverse=True)
    partial.sort(key=lambda x: x["weight"], reverse=True)
    weak_or_missing.sort(key=lambda x: x["weight"], reverse=True)

    return {
        "strong": strong,
        "partial": partial,
        "weak_or_missing": weak_or_missing,
    }

def _safe_join_summary(resume: StructuredResume) -> str:
    return "\n".join(resume.summary or [])

def _format_experience_for_prompt(resume: StructuredResume) -> str:
    lines = []
    for exp in resume.experience:
        header = " / ".join([x for x in [exp.title, exp.organization, exp.location, exp.period] if x])
        if header:
            lines.append(f"- {header}")
        for b in exp.bullets or []:
            lines.append(f"  • {b}")
        lines.append("")  # blank line between roles
    return "\n".join(lines).strip()

def _format_projects_for_prompt(resume: StructuredResume) -> str:
    lines = []
    for proj in resume.projects:
        header = " / ".join([x for x in [proj.title, ", ".join(proj.tech or []), proj.period] if x])
        if header:
            lines.append(f"- {header}")
        for b in proj.bullets or []:
            lines.append(f"  • {b}")
        lines.append("")
    return "\n".join(lines).strip()

def _extract_top_keywords(gap: Dict[str, List[Dict[str, Any]]],
                          max_per_bucket: int = 8) -> Dict[str, List[str]]:
    """
    Select a compact set of keywords to optimize around.
    """
    return {
        "strong": [r["skill"] for r in gap["strong"][:max_per_bucket]],
        "partial": [r["skill"] for r in gap["partial"][:max_per_bucket]],
        "weak_or_missing": [r["skill"] for r in gap["weak_or_missing"][:max_per_bucket]],
    }

def _call_llm_for_optimization(summary_text: str,
                               experience_text: str,
                               projects_text: str,
                               jd_title: str,
                               top_keywords: Dict[str, List[str]]) -> Dict[str, Any]:
    """
    Single LLM call that returns JSON with:
      - summary_phrasings: candidate phrases to integrate in summary
      - experience_phrasings: bullet-level rewrite ideas
      - project_phrasings: bullet-level rewrite ideas
      - keyword_synonyms: ATS-friendly variants / expansions
    We ASK for JSON only; we still validate/parsing on our side.
    """
    prompt = f"""
You are an ATS-focused résumé editor.

Your job:
1. Read the candidate's current résumé sections.
2. Study the job title and the JD-aligned keywords.
3. Suggest natural, non-obvious ways to weave these keywords into the résumé
   WITHOUT fabricating experience and WITHOUT changing the overall structure.

Return ONLY valid JSON with this shape:

{{
  "summary_phrasings": [
    {{
      "purpose": "subtle emphasis" | "strong emphasis",
      "suggested_line": "single sentence that could replace or precede one summary line"
    }}
  ],
  "experience_phrasings": [
    {{
      "hint": "short description of which kind of bullet this fits (e.g., 'cloud data pipelines role')",
      "suggested_bullet": "rewritten bullet that keeps meaning but better aligns with JD keywords"
    }}
  ],
  "project_phrasings": [
    {{
      "hint": "where this phrasing might be used",
      "suggested_bullet": "rewritten bullet for a project, preserving original intent"
    }}
  ],
  "keyword_synonyms": [
    {{
      "target_keyword": "one of the JD skills",
      "variants": ["short synonym or phrasing", "..."]
    }}
  ]
}}

Constraints:
- Do NOT invent new tools, frameworks, roles, or responsibilities.
- Prefer rewording over adding new content.
- Phrasings must sound human and subtle, not like keyword stuffing.
- You may reference multiple related keywords in a single phrase.

---- CONTEXT START ----
JOB_TITLE: {jd_title}

JD_KEYWORDS_STRONG: {top_keywords["strong"]}
JD_KEYWORDS_PARTIAL: {top_keywords["partial"]}
JD_KEYWORDS_WEAK_OR_MISSING: {top_keywords["weak_or_missing"]}

CURRENT_SUMMARY:
\"\"\"
{summary_text}
\"\"\"

CURRENT_EXPERIENCE (headers + bullets):
\"\"\"
{experience_text}
\"\"\"

CURRENT_PROJECTS (headers + bullets):
\"\"\"
{projects_text}
\"\"\"
---- CONTEXT END ----
""".strip()

    resp = hf_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are a precise JSON-only ATS optimization assistant."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=900,
        temperature=0.4,
    )
    raw = resp.choices[0].message.content

    # Minimal JSON extraction
    try:
        first = raw.find("{")
        last = raw.rfind("}")
        if first != -1 and last != -1:
            raw_json = raw[first:last+1]
            return json.loads(raw_json)
        # fallback if the whole response is just JSON
        return json.loads(raw)
    except Exception:
        # if parsing fails, return a safe empty structure
        return {
            "summary_phrasings": [],
            "experience_phrasings": [],
            "project_phrasings": [],
            "keyword_synonyms": [],
            "raw_response": raw,
        }
import hashlib
from pathlib import Path

CACHE_DIR = Path("/content/data/opt_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _fingerprint_for_block3(resume: StructuredResume,
                            jd: JDFeatureMap,
                            alignment_report: Dict[str, Any]) -> str:
    """
    Stable fingerprint of the inputs that drive Block 3.
    If resume/JD/alignment don’t change, we’ll reuse cached suggestions.
    """
    payload = {
        "resume": resume.model_dump(),
        "jd": jd.model_dump(),
        "alignment": alignment_report,
        "model": LLM_MODEL,
    }
    s = json.dumps(payload, sort_keys=True)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def run_block3_keyword_optimization(resume: StructuredResume,
                                    jd: JDFeatureMap,
                                    alignment_report: Dict[str, Any]) -> Dict[str, Any]:
    """
    Orchestrate Block 3 with on-disk caching.
    If the same resume + JD + alignment are passed again,
    we load the previous suggestions instead of calling the LLM.
    """
    fp = _fingerprint_for_block3(resume, jd, alignment_report)
    cache_path = CACHE_DIR / f"{fp}.json"

    # 1) Try cache
    if cache_path.exists():
        print("🔁 Block 3: loaded optimization bundle from cache.")
        return json.loads(cache_path.read_text())

    # 2) Compute fresh
    gap = build_gap_summary(alignment_report)
    top_keywords = _extract_top_keywords(gap)

    summary_text = _safe_join_summary(resume)
    experience_text = _format_experience_for_prompt(resume)
    projects_text = _format_projects_for_prompt(resume)

    llm_suggestions = _call_llm_for_optimization(
        summary_text=summary_text,
        experience_text=experience_text,
        projects_text=projects_text,
        jd_title=jd.job_title or "Software Engineer",
        top_keywords=top_keywords,
    )

    optimization_bundle = {
        "gap_summary": gap,
        "top_keywords": top_keywords,
        "llm_suggestions": llm_suggestions,
    }


    # 3) Save to cache for future runs
    cache_path.write_text(json.dumps(optimization_bundle, indent=2))
    print(f"💾 Block 3: cached optimization bundle to {cache_path}")

    return optimization_bundle

In [ ]:
optimization_bundle = run_block3_keyword_optimization(resume_struct, jd_map, alignment2)

gap = optimization_bundle["gap_summary"]
tops = optimization_bundle["top_keywords"]
llm_sug = optimization_bundle["llm_suggestions"]

print("=== GAP SUMMARY (TOP 5 EACH) ===")
print("Strong matches:")
for r in gap["strong"][:5]:
    print(f"  - {r['skill']} [{r['category']}] w={r['weight']} score={r['score']:.2f}")

print("\nPartial matches:")
for r in gap["partial"][:5]:
    print(f"  - {r['skill']} [{r['category']}] w={r['weight']} score={r['score']:.2f}")

print("\nWeak / Missing:")
for r in gap["weak_or_missing"][:5]:
    print(f"  - {r['skill']} [{r['category']}] w={r['weight']} score={r['score']:.2f}")

print("\n=== TOP KEYWORDS SELECTED FOR OPTIMIZATION ===")
print(json.dumps(tops, indent=2))

print("\n=== SAMPLE LLM SUMMARY PHRASINGS (FIRST 3) ===")
for p in llm_sug.get("summary_phrasings", [])[:3]:
    print(f"- [{p.get('purpose','?')}] {p.get('suggested_line','')}")

print("\n=== SAMPLE EXPERIENCE PHRASINGS (FIRST 3) ===")
for p in llm_sug.get("experience_phrasings", [])[:3]:
    print(f"- hint: {p.get('hint','')}  | bullet: {p.get('suggested_bullet','')}")

print("\n=== SAMPLE PROJECT PHRASINGS (FIRST 3) ===")
for p in llm_sug.get("project_phrasings", [])[:3]:
    print(f"- hint: {p.get('hint','')}  | bullet: {p.get('suggested_bullet','')}")

# **Block 4A: Skills Optimization and Intelligent Placement**

This block handles skills specifically.

After Block 2 identifies weak or missing but important JD skills, Block 4A shows those recommended skills to the user and lets them choose which ones they actually want to add.

That part is important because skills should not be added automatically. The user should remain in control.

Once the user selects the skills, the LLM is used again, but in a narrow way. Its job is only to decide where each new skill should go in the existing skills section.

It can either:

place the skill into an existing section
or suggest a new section if needed

Then the selected skills are merged into the structured resume.

I also fixed this block so that:

duplicate skill recommendations are removed
duplicate user selections are removed
LLM outputs wrapped in markdown fences still parse correctly
skill placements are applied cleanly into the resume object

So Block 4A is the structured skill enrichment stage.

It improves alignment while still preserving user control and logical section organization.

In [ ]:
# === LAYER 3 — BLOCK 4A: Skills Optimization & Intelligent Placement (FIXED) ===

from typing import List, Dict, Any
from copy import deepcopy
import json
import os
import re
from huggingface_hub import InferenceClient

LLAMA_MODEL = "meta-llama/Llama-3.1-70B-Instruct"
llm_client = InferenceClient(
    model=LLAMA_MODEL,
    api_key=os.environ.get("HF_TOKEN"),
    timeout=80,
)

# -------------------------------------------------
# 1) Helper: extract recommended skills (JD-driven)
# -------------------------------------------------
def get_recommended_skills(alignment_report: Dict[str, Any],
                           min_weight: float = 0.3) -> List[Dict[str, Any]]:
    """
    Pull recommended skills from alignment report and deduplicate by skill name.
    Keeps the highest-weight version if the same skill appears under multiple categories.
    """
    rec_names = set(alignment_report.get("recommendations", []))
    best_by_skill = {}

    for m in alignment_report.get("skill_matches", []):
        skill = (m.get("skill") or "").strip()
        if not skill:
            continue
        if skill not in rec_names:
            continue
        if m.get("jd_weight", 0.0) < min_weight:
            continue

        key = skill.lower()
        candidate = {
            "skill": skill,
            "category": m.get("category"),
            "jd_weight": m.get("jd_weight", 0.0),
            "match_score": m.get("match_score", 0.0),
        }

        # keep the strongest weight if duplicates exist
        if key not in best_by_skill or candidate["jd_weight"] > best_by_skill[key]["jd_weight"]:
            best_by_skill[key] = candidate

    out = list(best_by_skill.values())
    out.sort(key=lambda x: x["jd_weight"], reverse=True)
    return out

# -------------------------------------------------
# 2) Prompt Builder
# -------------------------------------------------
def _build_skills_prompt(existing_sections: List[Dict[str, Any]],
                         new_skills: List[str]) -> str:
    sections_json = json.dumps(existing_sections, indent=2)
    skills_json = json.dumps(new_skills, indent=2)

    return f"""
You are helping organize skills inside a resume.

You will receive:
1) Existing SKILL SECTIONS from the resume
2) NEW SKILLS that should be added

Your job:
- For EACH new skill, decide exactly ONE target section
- Prefer using an existing section if it fits naturally
- Create a new section only if absolutely necessary
- Keep new section names short and natural for a resume

Return ONLY valid JSON in this exact format:

{{
  "placements": [
    {{
      "skill": "<skill exactly as given>",
      "target_section": "<existing section name OR NEW:Section Name>"
    }}
  ]
}}

Existing sections:
{sections_json}

Skills to place:
{skills_json}
""".strip()

# -------------------------------------------------
# 3) JSON cleaning helper
# -------------------------------------------------
def _extract_json_object(raw: str) -> Dict[str, Any]:
    """
    Robustly parse JSON even if the model wraps it in ```json fences
    or adds extra commentary.
    """
    txt = (raw or "").strip()

    # remove fenced markdown blocks if present
    fence_match = re.search(r"```(?:json)?\s*([\s\S]*?)```", txt, re.IGNORECASE)
    if fence_match:
        txt = fence_match.group(1).strip()

    # if still not plain JSON, extract first {...} block
    if not txt.startswith("{"):
        obj_match = re.search(r"\{[\s\S]*\}", txt)
        if obj_match:
            txt = obj_match.group(0).strip()

    return json.loads(txt)

# -------------------------------------------------
# 4) LLM call
# -------------------------------------------------
def llm_decide_skill_placement(existing_sections: List[Dict[str, Any]],
                               new_skills: List[str]) -> List[Dict[str, str]]:
    if not new_skills:
        return []

    # dedupe incoming skills while preserving order
    deduped_skills = []
    seen = set()
    for s in new_skills:
        skill = (s or "").strip()
        key = skill.lower()
        if skill and key not in seen:
            seen.add(key)
            deduped_skills.append(skill)

    prompt = _build_skills_prompt(existing_sections, deduped_skills)

    resp = llm_client.chat.completions.create(
        model=LLAMA_MODEL,
        messages=[
            {"role": "system", "content": "You output ONLY valid JSON. No markdown fences. No explanations."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=600,
        temperature=0.0,
    )

    raw = resp.choices[0].message.content.strip()

    try:
        data = _extract_json_object(raw)
        placements = data.get("placements", [])

        # basic cleanup + dedupe by skill
        cleaned = []
        seen = set()
        for p in placements:
            skill = (p.get("skill") or "").strip()
            target = (p.get("target_section") or "").strip()
            if not skill or not target:
                continue

            key = skill.lower()
            if key in seen:
                continue
            seen.add(key)

            cleaned.append({
                "skill": skill,
                "target_section": target
            })

        return cleaned

    except Exception as e:
        print("⚠️ Non-JSON or invalid JSON returned:")
        print(raw)
        print(f"Parse error: {e}")
        return []

# -------------------------------------------------
# 5) Apply placements
# -------------------------------------------------
def apply_skill_placements(resume: StructuredResume,
                           placements: List[Dict[str, str]]) -> StructuredResume:
    updated = deepcopy(resume)

    name_to_group = {
        (sg.category or f"Section_{i}"): sg
        for i, sg in enumerate(updated.skills)
    }

    for p in placements:
        skill = (p.get("skill") or "").strip()
        target = (p.get("target_section") or "").strip()

        if not skill or not target:
            continue

        if target.startswith("NEW:"):
            sec_name = target.replace("NEW:", "", 1).strip() or "Additional Skills"
        else:
            sec_name = target

        if sec_name not in name_to_group:
            grp = SkillGroup(category=sec_name, items=[])
            updated.skills.append(grp)
            name_to_group[sec_name] = grp
        else:
            grp = name_to_group[sec_name]

        # add only if not already present
        if not any(skill.lower() == s.lower() for s in grp.items):
            grp.items.append(skill)

    return updated

# -------------------------------------------------
# 6) DRIVER (WRAPPED WITH UNIFIED OUTPUT FORMAT)
# -------------------------------------------------
def run_skills_optimization(resume: StructuredResume,
                            jd_map: JDFeatureMap,
                            alignment_report: Dict[str, Any]) -> Dict[str, Any]:

    recs = get_recommended_skills(alignment_report, min_weight=0.3)
    if not recs:
        print("No skill recommendations.")
        return {
            "updated_resume": resume,
            "skills_added": [],
            "placements": [],
            "updated_sections_preview": []
        }

    print("\n=== RECOMMENDED SKILLS ===")
    for i, r in enumerate(recs, 1):
        print(f"{i}. {r['skill']}  |  {r['category']}  | JD weight={r['jd_weight']:.3f}")

    raw = input("\nSelect skills to add (1,3,5) or ENTER to skip:\n> ").strip()
    if not raw:
        print("Skipping skill additions.")
        return {
            "updated_resume": resume,
            "skills_added": [],
            "placements": [],
            "updated_sections_preview": []
        }

    idxs = sorted(set([
        int(x.strip()) - 1
        for x in raw.split(",")
        if x.strip().isdigit() and 1 <= int(x.strip()) <= len(recs)
    ]))

    if not idxs:
        print("No valid skill selections. Skipping.")
        return {
            "updated_resume": resume,
            "skills_added": [],
            "placements": [],
            "updated_sections_preview": []
        }

    # dedupe chosen skills
    chosen = []
    seen = set()
    for i in idxs:
        sk = (recs[i]["skill"] or "").strip()
        key = sk.lower()
        if sk and key not in seen:
            seen.add(key)
            chosen.append(sk)

    print("\nYou selected:")
    for sk in chosen:
        print(" -", sk)

    # snapshot existing sections
    existing_sections = [
        {"name": sg.category or "Skills", "items": sg.items or []}
        for sg in resume.skills
    ]

    print("\n🔎 Calling LLM for placement...")
    placements = llm_decide_skill_placement(existing_sections, chosen)

    print("\nLLM Decisions:")
    if placements:
        for p in placements:
            print(f"  {p['skill']} → {p['target_section']}")
    else:
        print("  (No valid placements returned)")

    updated_resume = apply_skill_placements(resume, placements)

    preview = [
        {"section": sg.category, "items": sg.items}
        for sg in updated_resume.skills
    ]

    print("\n=== UPDATED SKILLS PREVIEW ===")
    for row in preview:
        print(f"\n{row['section']}:")
        print("  " + ", ".join(row["items"]))

    return {
        "updated_resume": updated_resume,
        "skills_added": chosen,
        "placements": placements,
        "updated_sections_preview": preview
    }

# -------------------------------------------------
# 7) RUN BLOCK 4A
# -------------------------------------------------
print("\n\n=== RUNNING BLOCK 4A: SKILLS OPTIMIZATION ===")
skill_block_output = run_skills_optimization(resume_struct, jd_map, alignment2)

# **Block 4B: Apply LLM Suggestions to Summary, Experience, and Projects**

This is the selective editing block.

Block 3 generated suggestions, but Block 4B is where those suggestions are actually reviewed and applied.

It has three parts.

Summary

The system builds up to three summary variants using the LLM suggestions plus the existing summary lines.

The goal here is to keep the original tone and structure familiar while nudging the content closer to the JD.

Then the user gets to choose which summary variant to apply, or keep the original.

Experience

For each experience suggestion, the system matches the LLM hint to one experience entry, shows the current bullets, shows the suggested rewrite, and lets the user decide whether to:

replace a bullet
append a new bullet
or skip it

So the model suggests, but the user decides.

Projects

Projects work the same way as experience.

The system matches the suggestion to the right project, shows the old bullets and suggested bullet, and lets the user choose whether to apply it.

So Block 4B is the human-in-the-loop refinement stage.

It makes sure the edits are not blindly accepted.

In [ ]:
# === LAYER 3 — BLOCK 4B: Apply LLM Suggestions to Summary / Experience / Projects ===
import json
from typing import Dict, Any, List, Optional

# ------------- Helpers for SUMMARY -----------------

def build_summary_variants_from_llm(
    resume: StructuredResume,
    llm_suggestions: Dict[str, Any],
    max_variants: int = 3,
) -> List[List[str]]:
    """
    Build up to 3 summary variants using:
      - LLM 'summary_phrasings' (subtle / strong, etc.)
      - Existing summary lines, to keep tone/format familiar.
    Returns: list of variants, each variant = list[str] (lines).
    """
    orig = resume.summary or []
    phr = llm_suggestions.get("summary_phrasings", []) or []

    subtle_line: Optional[str] = None
    strong_line: Optional[str] = None
    generic_line: Optional[str] = None

    # Try to identify subtle / strong by 'purpose', fallback to first entries
    for p in phr:
        purpose = (p.get("purpose") or "").lower()
        line = p.get("suggested_line", "").strip()
        if not line:
            continue
        if "subtle" in purpose and subtle_line is None:
            subtle_line = line
        elif "strong" in purpose and strong_line is None:
            strong_line = line
        elif generic_line is None:
            generic_line = line

    if generic_line is None and phr:
        generic_line = phr[0].get("suggested_line", "").strip() or None

    variants: List[List[str]] = []

    # Variant 1: subtle lead + rest of original (format preserved)
    if subtle_line:
        v1 = [subtle_line]
        if len(orig) > 1:
            v1.extend(orig[1:])
        variants.append(v1)

    # Variant 2: strong lead + rest of original
    if strong_line and len(variants) < max_variants:
        v2 = [strong_line]
        if len(orig) > 1:
            v2.extend(orig[1:])
        variants.append(v2)

    # Variant 3: blended variant (generic or subtle+strong) + one key original line
    if len(variants) < max_variants and (generic_line or subtle_line or strong_line):
        v3: List[str] = []
        lead = generic_line or subtle_line or strong_line
        if lead:
            v3.append(lead)
        # keep one key original line (often AWS / pipelines)
        if orig:
            # pick the line that mentions AWS or cloud if present
            key_line = None
            for line in orig:
                low = line.lower()
                if "aws" in low or "cloud" in low:
                    key_line = line
                    break
            if key_line is None:
                key_line = orig[min(1, len(orig)-1)]  # second line if exists
            if key_line:
                v3.append(key_line)
        variants.append(v3)

    # Fallback: if nothing built, just return existing summary as single variant
    if not variants:
        variants.append(orig)

    return variants[:max_variants]


def choose_and_apply_summary_variant(
    resume: StructuredResume,
    variants: List[List[str]]
) -> Dict[str, Any]:
    """
    Interactively show summary variants, let user pick 1–N,
    then update resume.summary and return change-log fragment.
    """
    print("=== ORIGINAL SUMMARY ===")
    for line in (resume.summary or []):
        print(line)
    print("\n=== GENERATED SUMMARY VARIANTS ===\n")

    for i, v in enumerate(variants, 1):
        print(f"--- Variant {i} ---")
        for line in v:
            print(line)
        print()

    choice_raw = input("Select summary variant to apply (1/2/3 or ENTER to keep original): ").strip()
    if not choice_raw:
        print("No variant selected. Keeping original summary.")
        return {
            "old": resume.summary,
            "new": resume.summary,
            "variant_selected": None
        }

    try:
        idx = int(choice_raw) - 1
    except ValueError:
        print("Invalid input. Keeping original summary.")
        return {
            "old": resume.summary,
            "new": resume.summary,
            "variant_selected": None
        }

    if idx < 0 or idx >= len(variants):
        print("Index out of range. Keeping original summary.")
        return {
            "old": resume.summary,
            "new": resume.summary,
            "variant_selected": None
        }

    old_summary = list(resume.summary or [])
    new_summary = variants[idx]

    resume.summary = new_summary  # mutate in place

    print("\n✅ Summary updated.\n")
    return {
        "old": old_summary,
        "new": new_summary,
        "variant_selected": idx + 1
    }


# ------------- Helpers for EXPERIENCE -----------------

def _find_experience_index_by_hint(resume: StructuredResume, hint: str) -> Optional[int]:
    """Map an LLM 'hint' string to one experience entry index."""
    hint_low = (hint or "").lower()
    for i, exp in enumerate(resume.experience):
        title_low = (exp.title or "").lower()
        org_low = (exp.organization or "").lower()
        if title_low and title_low in hint_low:
            return i
        if org_low and org_low in hint_low:
            return i
    return None


def apply_experience_suggestions(
    resume: StructuredResume,
    llm_suggestions: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """
    For each experience suggestion, show:
      - role + existing bullets
      - suggested bullet
    User decides:
      - replace a bullet (enter index),
      - append as new bullet ('a'),
      - or skip (ENTER).
    Returns list of change-log entries for experience edits.
    """
    changes: List[Dict[str, Any]] = []
    exp_sugs = llm_suggestions.get("experience_phrasings", []) or []

    if not exp_sugs:
        print("\n(No experience suggestions from LLM.)")
        return changes

    print("\n=== EXPERIENCE BULLET REFINEMENT ===")
    for sug in exp_sugs:
        hint = sug.get("hint", "")
        new_bullet = sug.get("suggested_bullet", "").strip()
        if not new_bullet:
            continue

        idx = _find_experience_index_by_hint(resume, hint)
        if idx is None or idx < 0 or idx >= len(resume.experience):
            print(f"\nSkipping suggestion with hint '{hint}' — no matching experience entry found.")
            continue

        exp = resume.experience[idx]
        print("\n------------------------------------------")
        print(f"Role [{idx}]: {exp.title or ''}  |  {exp.organization or ''}")
        print("Current bullets:")
        for j, b in enumerate(exp.bullets):
            print(f"  [{j}] {b}")
        print("\nSuggested bullet:")
        print(f"  >>> {new_bullet}\n")

        choice = input(
            "Action? (enter index to replace that bullet, 'a' to append, ENTER to skip): "
        ).strip().lower()

        if not choice:
            print("  → Skipped.")
            continue

        entry_change: Dict[str, Any] = {
            "entry_index": idx,
            "entry_title": exp.title,
            "changes": []
        }

        if choice == "a":
            exp.bullets.append(new_bullet)
            entry_change["changes"].append({
                "bullet_index": len(exp.bullets) - 1,
                "old": None,
                "new": new_bullet
            })
            print("  ✅ Appended as new bullet.")
        else:
            try:
                j = int(choice)
            except ValueError:
                print("  Invalid input. Skipping.")
                continue
            if j < 0 or j >= len(exp.bullets):
                print("  Index out of range. Skipping.")
                continue
            old_b = exp.bullets[j]
            exp.bullets[j] = new_bullet
            entry_change["changes"].append({
                "bullet_index": j,
                "old": old_b,
                "new": new_bullet
            })
            print("  ✅ Replaced bullet.")

        if entry_change["changes"]:
            changes.append(entry_change)

    return changes


# ------------- Helpers for PROJECTS -----------------

def _find_project_index_by_hint(resume: StructuredResume, hint: str) -> Optional[int]:
    """Map an LLM 'hint' string to one project index."""
    hint_low = (hint or "").lower()
    for i, proj in enumerate(resume.projects):
        title_low = (proj.title or "").lower()
        if title_low and title_low in hint_low:
            return i
    return None


def apply_project_suggestions(
    resume: StructuredResume,
    llm_suggestions: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """
    Similar to experience: for each project suggestion,
    show current bullets and suggested bullet and let user choose.
    """
    changes: List[Dict[str, Any]] = []
    proj_sugs = llm_suggestions.get("project_phrasings", []) or []

    if not proj_sugs:
        print("\n(No project suggestions from LLM.)")
        return changes

    print("\n=== PROJECT BULLET REFINEMENT ===")
    for sug in proj_sugs:
        hint = sug.get("hint", "")
        new_bullet = sug.get("suggested_bullet", "").strip()
        if not new_bullet:
            continue

        idx = _find_project_index_by_hint(resume, hint)
        if idx is None or idx < 0 or idx >= len(resume.projects):
            print(f"\nSkipping project suggestion with hint '{hint}' — no matching project found.")
            continue

        proj = resume.projects[idx]
        print("\n------------------------------------------")
        print(f"Project [{idx}]: {proj.title or ''}")
        print("Current bullets:")
        for j, b in enumerate(proj.bullets):
            print(f"  [{j}] {b}")
        print("\nSuggested bullet:")
        print(f"  >>> {new_bullet}\n")

        choice = input(
            "Action? (enter index to replace that bullet, 'a' to append, ENTER to skip): "
        ).strip().lower()

        if not choice:
            print("  → Skipped.")
            continue

        entry_change: Dict[str, Any] = {
            "project_index": idx,
            "project_title": proj.title,
            "changes": []
        }

        if choice == "a":
            proj.bullets.append(new_bullet)
            entry_change["changes"].append({
                "bullet_index": len(proj.bullets) - 1,
                "old": None,
                "new": new_bullet
            })
            print("  ✅ Appended as new project bullet.")
        else:
            try:
                j = int(choice)
            except ValueError:
                print("  Invalid input. Skipping.")
                continue
            if j < 0 or j >= len(proj.bullets):
                print("  Index out of range. Skipping.")
                continue
            old_b = proj.bullets[j]
            proj.bullets[j] = new_bullet
            entry_change["changes"].append({
                "bullet_index": j,
                "old": old_b,
                "new": new_bullet
            })
            print("  ✅ Replaced project bullet.")

        if entry_change["changes"]:
            changes.append(entry_change)

    return changes


# ------------- Orchestrator for BLOCK 4 -----------------

def run_block4_apply_llm_suggestions(
    resume: StructuredResume,
    llm_suggestions: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Orchestrates:
      1) Summary variants (LLM-based) + user choice
      2) Experience bullet refinement (per suggestion)
      3) Project bullet refinement (per suggestion)
    Returns a unified change-log dict in the agreed format.
    """
    change_log: Dict[str, Any] = {
        "summary": None,
        "experience": [],
        "projects": []
    }

    # 1) SUMMARY
    print("\n SUMMARY VARIANTS (LLM-BASED) ")
    summary_variants = build_summary_variants_from_llm(resume, llm_suggestions)
    summary_change = choose_and_apply_summary_variant(resume, summary_variants)
    change_log["summary"] = summary_change

    # 2) EXPERIENCE
    print("\n EXPERIENCE BULLET REFINEMENT (LLM-BASED)")
    exp_changes = apply_experience_suggestions(resume, llm_suggestions)
    change_log["experience"] = exp_changes

    # 3) PROJECTS
    print("\n PROJECT BULLET REFINEMENT (LLM-BASED)")
    proj_changes = apply_project_suggestions(resume, llm_suggestions)
    change_log["projects"] = proj_changes

    print("\n✅ BLOCK 4B COMPLETE — SUMMARY / EXPERIENCE / PROJECTS UPDATED IN MEMORY.")
    return change_log


# ---------------- RUN BLOCK 4 NOW ----------------
print("=== RUNNING BLOCK 4B WITH CURRENT resume_struct + llm_suggestions ===")
block4_change_log = run_block4_apply_llm_suggestions(resume_struct, llm_sug)

print("\n=== CHANGE LOG (JSON) ===")
print(json.dumps(block4_change_log, indent=2))

In [ ]:
def print_skill_block_output(skill_output):
    """
    Pretty-print the standardized Block 4B output structure.
    Automatically converts Pydantic models into JSON-safe dicts.
    """
    import json

    safe = {}

    for key, value in skill_output.items():
        if hasattr(value, "model_dump"):
            # Pydantic model
            safe[key] = value.model_dump()
        else:
            # Anything else (list, dict, str)
            safe[key] = value

    print(json.dumps(safe, indent=2))
    return safe

print_skill_block_output(skill_block_output)

# **Block 4C: Consolidation of All Edits**

By the time Block 4C runs, different parts of the resume have been updated in different places.

Skills came from Block 4A.
Summary, experience, and projects were edited in Block 4B.

So Block 4C consolidates everything into one final structured resume object.

It starts from the skill-updated resume, then copies over the updated summary, experience, and project content, and also builds one unified change log.

That change log records:

skills added
skill placements
summary changes
experience changes
project changes

This is useful because it gives the system an audit trail of exactly what changed.

So Block 4C is basically the merge-and-finalize step inside Layer

In [ ]:
# === LAYER 3 — BLOCK 4C: Consolidation of All Edits ===
from typing import Dict, Any
from copy import deepcopy
import json

def consolidate_layer3_outputs(
    original_resume: StructuredResume,
    skill_block: Dict[str, Any],
    block4_changes: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Combines:
      - Skill additions / placements
      - Summary variant selection
      - Experience edits
      - Project edits
    And outputs:
      - final resume (StructuredResume)
      - unified change log for audit
    """

    # 1. Start with skill-updated resume (Block 4A)
    final_resume = skill_block["updated_resume"]

    # 2. Summary was modified IN-PLACE in original resume struct used for Block 4B
    #    We now copy that final summary into final_resume
    final_resume.summary = deepcopy(original_resume.summary)

    # 3. Copy updated experience (mutated by Block 4B)
    final_resume.experience = deepcopy(original_resume.experience)

    # 4. Copy updated projects (mutated by Block 4B)
    final_resume.projects = deepcopy(original_resume.projects)

    # 5. Build unified change log
    unified_log = {
        "skills_added": skill_block["skills_added"],
        "skill_placements": skill_block["placements"],
        "summary_changes": block4_changes.get("summary"),
        "experience_changes": block4_changes.get("experience"),
        "project_changes": block4_changes.get("projects")
    }

    # 6. Return final resume and changes
    return {
        "final_resume": final_resume,
        "change_log": unified_log
    }


# ------------------- RUN BLOCK 4C -------------------

layer3_output = consolidate_layer3_outputs(
    original_resume=resume_struct,    # mutated by Block 4B
    skill_block=skill_block_output,   # result of 4A
    block4_changes=block4_change_log  # result of 4B
)

print("\n=== BLOCK 4C — CONSOLIDATION COMPLETE ===")
print("Final resume ready for export.")

# **Block 5: Template-Aware Final Export**

Once the optimized resume is ready, Block 5 prepares it for export.

The important idea here is that export is not done in a generic way.
Instead, it tries to respect the formatting template extracted earlier from the original resume.

So Block 5 does not just dump the structured resume into a default layout. It uses template cues such as:

section heading style
section order
bullet style
project layout
skills layout
contact separator
density and page compactness
education and experience layout style

It builds a layout representation first, then renders that layout into different formats:

plain text
DOCX
PDF
LaTeX

The DOCX exporter is the strongest one because it supports compact alignment, summary as a paragraph block, two-column header/date structure, and closer visual preservation of the original resume shell.

The PDF exporter uses a simpler renderer and is more approximate, which is why DOCX came out better in your tests.

So Block 5 is the final assembly and export layer of the optimization pipeline.

In [ ]:
!pip install reportlab python-docx

In [ ]:
# === LAYER 3 — BLOCK 5: TEMPLATE-AWARE FINAL EXPORT (FINAL) ===

import os
import json
from typing import List, Dict, Any, Optional

# Optional DOCX imports
try:
    from docx import Document
    from docx.shared import Pt, Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    Document = None
    WD_ALIGN_PARAGRAPH = None

# Optional PDF imports
try:
    from reportlab.lib.pagesizes import LETTER
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT
    from reportlab.lib.units import inch
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
except ImportError:
    SimpleDocTemplate = None
    Table = None
    TableStyle = None

# ---------------------------------------------
# 0) Decide which resume object is final
# ---------------------------------------------
if "layer3_output" in globals() and isinstance(layer3_output, dict) and "final_resume" in layer3_output:
    final_resume_struct = layer3_output["final_resume"]
else:
    final_resume_struct = skill_block_output.get("updated_resume", resume_struct)

# ---------------------------------------------
# 1) Load template profile from Layer 1
# ---------------------------------------------
def load_template_profile(session_id: str) -> Dict[str, Any]:
    path = f"data/{session_id}.template.json"
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)

    return {
        "section_heading_case": "uppercase",
        "section_order": [
            "SUMMARY",
            "EDUCATION",
            "TECHNICAL SKILLS",
            "WORK EXPERIENCE",
            "PROJECTS",
            "CERTIFICATIONS"
        ],
        "bullet_style": "•",
        "skills_layout": "category_colon_list",
        "project_layout": "pipe_format",
        "date_format": "human_month_year",
        "has_blank_line_spacing": False,
        "density": "medium",
        "experience_layout": "two_line_header",
        "education_layout": "two_line_header",
        "contact_separator": "•",
        "estimated_line_count": 60,
        "page_density": "one_page_compact"
    }

# ---------------------------------------------
# 2) Template helpers
# ---------------------------------------------
def section_title(title: str, template: Dict[str, Any]) -> str:
    return title.upper() if template.get("section_heading_case") == "uppercase" else title.title()

def bullet_symbol(template: Dict[str, Any]) -> str:
    return template.get("bullet_style", "•")

def contact_separator(template: Dict[str, Any]) -> str:
    sep = template.get("contact_separator", "•")
    if sep == "space":
        return " "
    return f" {sep} "

def style_profile(template: Dict[str, Any]) -> Dict[str, Any]:
    """
    Compact rendering profile driven by detected template density.
    """
    density = template.get("density", "medium")
    page_density = template.get("page_density", "one_page_compact")

    cfg = {
        "base_font_size": 10.5,
        "name_font_size": 15,
        "contact_font_size": 9.5,
        "section_font_size": 11,
        "line_leading": 11.2,
        "section_space_before": 3,
        "section_space_after": 1,
        "entry_space_after": 0,
        "bullet_indent_in": 0.18,
        "margins_in": 0.48
    }

    if page_density != "one_page_compact":
        cfg.update({
            "base_font_size": 11,
            "name_font_size": 16,
            "contact_font_size": 10,
            "section_font_size": 12,
            "line_leading": 12.5,
            "section_space_before": 5,
            "section_space_after": 2,
            "entry_space_after": 1,
            "bullet_indent_in": 0.20,
            "margins_in": 0.60
        })

    if density == "compact":
        cfg["line_leading"] = max(10.8, cfg["line_leading"] - 0.2)
    elif density == "spacious":
        cfg["line_leading"] += 0.5

    return cfg

def build_contact_line(resume: StructuredResume, template: Dict[str, Any]) -> str:
    parts = []
    if resume.contact.location:
        parts.append(resume.contact.location)
    if resume.contact.phone:
        parts.append(resume.contact.phone)
    if resume.contact.email:
        parts.append(resume.contact.email)
    if resume.contact.links:
        parts.extend([ln.strip() for ln in resume.contact.links if ln and ln.strip()])
    return contact_separator(template).join(parts)

# ---------------------------------------------
# 3) Section-aware date rendering
# ---------------------------------------------
def _extract_year(val: Optional[str]) -> Optional[str]:
    if not val:
        return None
    import re
    m = re.search(r"\b(19|20)\d{2}\b", val)
    return m.group(0) if m else None

def render_period_for_section(
    start_date: Optional[str],
    end_date: Optional[str],
    period: Optional[str],
    section_type: str,
    template: Dict[str, Any]
) -> str:
    """
    Projects stay compact like the original resume.
    Education/Experience prefer human-readable month-year if available.
    """
    if section_type == "projects":
        sy = _extract_year(start_date)
        ey = _extract_year(end_date)

        if sy and ey:
            return sy if sy == ey else f"{sy} – {ey}"
        if sy:
            return sy
        if period:
            py = _extract_year(period)
            return py or period
        return ""

    # education / experience
    if start_date and end_date:
        return f"{start_date} – {end_date}"
    if start_date and not end_date:
        return f"{start_date} – Present"
    if period:
        return period
    return ""

# ---------------------------------------------
# 4) Build section map, then order by template
# ---------------------------------------------
def build_layout(resume: StructuredResume, template: Dict[str, Any]) -> Dict[str, Any]:
    layout: Dict[str, Any] = {
        "name": resume.name or "",
        "contact_line": build_contact_line(resume, template),
        "sections": []
    }

    section_map: Dict[str, Dict[str, Any]] = {}

    # SUMMARY
    if resume.summary:
        section_map["SUMMARY"] = {
            "title": section_title("SUMMARY", template),
            "type": "summary",
            "lines": resume.summary
        }

    # EDUCATION
    if resume.education:
        entries = []
        for ed in resume.education:
            date_text = render_period_for_section(ed.start_date, ed.end_date, ed.period, "education", template)
            bullets = []
            if ed.coursework:
                bullets.append("Relevant Coursework: " + ", ".join(ed.coursework))
            bullets.extend(ed.bullets or [])

            entries.append({
                "line1_left": ed.institution or "",
                "line1_right": date_text,
                "line2_left": ed.degree or "",
                "line2_right": ed.location or "",
                "bullets": bullets
            })

        section_map["EDUCATION"] = {
            "title": section_title("EDUCATION", template),
            "type": "education",
            "entries": entries
        }

    # TECHNICAL SKILLS
    if resume.skills:
        skill_rows = []
        for sg in resume.skills:
            label = sg.category or "Skills"
            items = sg.items or []
            if not items:
                continue

            if template.get("skills_layout") == "category_colon_list":
                skill_rows.append({"label": label, "items": items})
            else:
                skill_rows.append({"label": None, "items": items})

        section_map["TECHNICAL SKILLS"] = {
            "title": section_title("TECHNICAL SKILLS", template),
            "type": "skills",
            "rows": skill_rows
        }

    # WORK EXPERIENCE
    if resume.experience:
        entries = []
        for e in resume.experience:
            date_text = render_period_for_section(e.start_date, e.end_date, e.period, "experience", template)

            entries.append({
                "line1_left": e.title or "",
                "line1_right": date_text,
                "line2_left": e.organization or "",
                "line2_right": e.location or "",
                "bullets": e.bullets or []
            })

        section_map["WORK EXPERIENCE"] = {
            "title": section_title("WORK EXPERIENCE", template),
            "type": "experience",
            "entries": entries
        }

    # PROJECTS
    if resume.projects:
        entries = []
        for p in resume.projects:
            date_text = render_period_for_section(p.start_date, p.end_date, p.period, "projects", template)
            tech_text = ", ".join(p.tech or [])

            left_text = " | ".join([x for x in [p.title, tech_text] if x]) \
                if template.get("project_layout") == "pipe_format" \
                else (p.title or "")

            entries.append({
                "line1_left": left_text,
                "line1_right": date_text,
                "bullets": p.bullets or []
            })

        section_map["PROJECTS"] = {
            "title": section_title("PROJECTS", template),
            "type": "projects",
            "entries": entries
        }

    # CERTIFICATIONS
    if resume.certifications:
        certs = []
        for c in resume.certifications:
            line = " | ".join([x for x in [c.title, c.issuer, c.year] if x])
            if line:
                certs.append(line)

        section_map["CERTIFICATIONS"] = {
            "title": section_title("CERTIFICATIONS", template),
            "type": "certifications",
            "bullets": certs
        }

    # preserve original order from template
    ordered_sections = []
    wanted_order = template.get("section_order", [])
    for sec_name in wanted_order:
        if sec_name in section_map:
            ordered_sections.append(section_map[sec_name])

    # append any unexpected sections at the end
    for sec_name, sec_obj in section_map.items():
        if sec_name not in wanted_order:
            ordered_sections.append(sec_obj)

    layout["sections"] = ordered_sections
    return layout

# ---------------------------------------------
# 5) Plain text renderer
# ---------------------------------------------
def render_plaintext(layout: Dict[str, Any], template: Dict[str, Any]) -> str:
    bullet = bullet_symbol(template)
    lines: List[str] = []

    if layout["name"]:
        lines.append(layout["name"])
        lines.append("")

    if layout["contact_line"]:
        lines.append(layout["contact_line"])
        lines.append("")

    for sec in layout["sections"]:
        lines.append(sec["title"])

        if sec["type"] == "summary":
            summary_text = " ".join(sec["lines"])
            lines.append(summary_text)

        elif sec["type"] == "skills":
            for row in sec["rows"]:
                if row["label"]:
                    lines.append(f"{row['label']}: {', '.join(row['items'])}")
                else:
                    lines.append(", ".join(row["items"]))

        elif sec["type"] in ("experience", "education", "projects"):
            for entry in sec["entries"]:
                line1_left = entry.get("line1_left", "")
                line1_right = entry.get("line1_right", "")
                header = " | ".join([x for x in [line1_left, line1_right] if x])
                if header:
                    lines.append(header)

                line2_left = entry.get("line2_left", "")
                line2_right = entry.get("line2_right", "")
                subheader = " | ".join([x for x in [line2_left, line2_right] if x])
                if subheader:
                    lines.append(subheader)

                for b in entry["bullets"]:
                    lines.append(f"{bullet} {b}")
                lines.append("")

        elif sec["type"] == "certifications":
            for b in sec["bullets"]:
                lines.append(f"{bullet} {b}")

        lines.append("")

    return "\n".join(lines).rstrip() + "\n"

# ---------------------------------------------
# 6) DOCX exporter
# ---------------------------------------------
def export_to_docx(layout: Dict[str, Any], template: Dict[str, Any], filename: str = "aligned_resume.docx") -> str:
    if Document is None or WD_ALIGN_PARAGRAPH is None:
        print("⚠️ python-docx not installed. Skipping DOCX export.")
        return ""

    cfg = style_profile(template)
    if template.get("page_density") == "one_page_compact":
        cfg["entry_space_after"] = 0

    bullet = bullet_symbol(template)
    doc = Document()

    # page margins
    for section in doc.sections:
        section.left_margin = Inches(cfg["margins_in"])
        section.right_margin = Inches(cfg["margins_in"])
        section.top_margin = Inches(cfg["margins_in"])
        section.bottom_margin = Inches(cfg["margins_in"])

    # base style
    style = doc.styles["Normal"]
    style.font.name = "Calibri"
    style.font.size = Pt(cfg["base_font_size"])

    def add_two_col_row(
        left_text: str,
        right_text: str,
        left_bold: bool = False,
        right_bold: bool = False,
        font_size: float = 11
    ):
        table = doc.add_table(rows=1, cols=2)
        table.autofit = False

        # widths tuned for compact one-page layout
        table.columns[0].width = Inches(5.8)
        table.columns[1].width = Inches(1.2)

        row = table.rows[0]
        left_cell = row.cells[0]
        right_cell = row.cells[1]

        p_left = left_cell.paragraphs[0]
        p_left.paragraph_format.space_before = Pt(0)
        p_left.paragraph_format.space_after = Pt(0)
        p_left.paragraph_format.line_spacing = 1.0
        r_left = p_left.add_run(left_text or "")
        r_left.bold = left_bold
        r_left.font.name = "Calibri"
        r_left.font.size = Pt(font_size)

        p_right = right_cell.paragraphs[0]
        p_right.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        p_right.paragraph_format.space_before = Pt(0)
        p_right.paragraph_format.space_after = Pt(0)
        p_right.paragraph_format.line_spacing = 1.0
        r_right = p_right.add_run(right_text or "")
        r_right.bold = right_bold
        r_right.font.name = "Calibri"
        r_right.font.size = Pt(font_size)

        # remove visible borders
        tbl = table._tbl
        for cell in row.cells:
            tc = cell._tc
            tcPr = tc.get_or_add_tcPr()
            # leave table invisible by not styling borders
            _ = tbl, tcPr

    # header
    if layout["name"]:
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(0)
        p.paragraph_format.line_spacing = 1.0
        run = p.add_run(layout["name"])
        run.bold = True
        run.font.name = "Calibri"
        run.font.size = Pt(cfg["name_font_size"])

    if layout["contact_line"]:
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.0
        run = p.add_run(layout["contact_line"])
        run.font.name = "Calibri"
        run.font.size = Pt(cfg["contact_font_size"])

    # sections
    for sec in layout["sections"]:
        hp = doc.add_paragraph()
        hp.paragraph_format.space_before = Pt(cfg["section_space_before"])
        hp.paragraph_format.space_after = Pt(cfg["section_space_after"])
        hp.paragraph_format.line_spacing = 1.0
        r = hp.add_run(sec["title"])
        r.bold = True
        r.font.name = "Calibri"
        r.font.size = Pt(cfg["section_font_size"])

        if sec["type"] == "summary":
            summary_text = " ".join(sec["lines"])
            p = doc.add_paragraph()
            p.paragraph_format.space_before = Pt(0)
            p.paragraph_format.space_after = Pt(0)
            p.paragraph_format.line_spacing = 1.0
            p.add_run(summary_text)

        elif sec["type"] == "skills":
            for row in sec["rows"]:
                p = doc.add_paragraph()
                p.paragraph_format.space_before = Pt(0)
                p.paragraph_format.space_after = Pt(0)
                p.paragraph_format.line_spacing = 1.0
                if row["label"]:
                    p.add_run(f"{row['label']}: ").bold = True
                    p.add_run(", ".join(row["items"]))
                else:
                    p.add_run(", ".join(row["items"]))

        elif sec["type"] in ("experience", "education"):
            for entry in sec["entries"]:
                add_two_col_row(
                    entry.get("line1_left", ""),
                    entry.get("line1_right", ""),
                    left_bold=True,
                    right_bold=False,
                    font_size=cfg["base_font_size"]
                )
                if entry.get("line2_left") or entry.get("line2_right"):
                    add_two_col_row(
                        entry.get("line2_left", ""),
                        entry.get("line2_right", ""),
                        left_bold=False,
                        right_bold=False,
                        font_size=cfg["base_font_size"]
                    )
                for b in entry["bullets"]:
                    pb = doc.add_paragraph()
                    pb.paragraph_format.left_indent = Inches(cfg["bullet_indent_in"])
                    pb.paragraph_format.space_before = Pt(0)
                    pb.paragraph_format.space_after = Pt(0)
                    pb.paragraph_format.line_spacing = 1.0
                    pb.add_run(f"{bullet} {b}")

        elif sec["type"] == "projects":
            for entry in sec["entries"]:
                add_two_col_row(
                    entry.get("line1_left", ""),
                    entry.get("line1_right", ""),
                    left_bold=True,
                    right_bold=False,
                    font_size=cfg["base_font_size"]
                )
                for b in entry["bullets"]:
                    pb = doc.add_paragraph()
                    pb.paragraph_format.left_indent = Inches(cfg["bullet_indent_in"])
                    pb.paragraph_format.space_before = Pt(0)
                    pb.paragraph_format.space_after = Pt(0)
                    pb.paragraph_format.line_spacing = 1.0
                    pb.add_run(f"{bullet} {b}")

        elif sec["type"] == "certifications":
            for b in sec["bullets"]:
                pb = doc.add_paragraph()
                pb.paragraph_format.left_indent = Inches(cfg["bullet_indent_in"])
                pb.paragraph_format.space_before = Pt(0)
                pb.paragraph_format.space_after = Pt(0)
                pb.paragraph_format.line_spacing = 1.0
                pb.add_run(f"{bullet} {b}")

    doc.save(filename)
    print(f"✅ DOCX written → {filename}")
    return filename

def export_to_pdf(layout: Dict[str, Any], template: Dict[str, Any], filename: str = "aligned_resume.pdf") -> str:
    if SimpleDocTemplate is None:
        print("⚠️ reportlab not installed. Skipping PDF export.")
        return ""

    cfg = style_profile(template)
    bullet = bullet_symbol(template)

    doc = SimpleDocTemplate(
        filename,
        pagesize=LETTER,
        leftMargin=cfg["margins_in"] * inch,
        rightMargin=cfg["margins_in"] * inch,
        topMargin=cfg["margins_in"] * inch,
        bottomMargin=cfg["margins_in"] * inch
    )

    styles = getSampleStyleSheet()

    name_style = ParagraphStyle(
        "NameStyle",
        parent=styles["Normal"],
        fontName="Helvetica-Bold",
        fontSize=cfg["name_font_size"],
        leading=cfg["line_leading"] + 1,
        spaceAfter=1,
        alignment=TA_LEFT
    )

    contact_style = ParagraphStyle(
        "ContactStyle",
        parent=styles["Normal"],
        fontName="Helvetica",
        fontSize=cfg["contact_font_size"],
        leading=cfg["line_leading"],
        spaceAfter=2,
        alignment=TA_LEFT
    )

    section_style = ParagraphStyle(
        "SectionStyle",
        parent=styles["Normal"],
        fontName="Helvetica-Bold",
        fontSize=cfg["section_font_size"],
        leading=cfg["line_leading"],
        spaceBefore=cfg["section_space_before"],
        spaceAfter=cfg["section_space_after"],
        alignment=TA_LEFT
    )

    body_style = ParagraphStyle(
        "BodyStyle",
        parent=styles["Normal"],
        fontName="Helvetica",
        fontSize=cfg["base_font_size"],
        leading=cfg["line_leading"],
        spaceAfter=0,
        alignment=TA_LEFT
    )

    bullet_style = ParagraphStyle(
        "BulletStyle",
        parent=body_style,
        leftIndent=cfg["bullet_indent_in"] * inch,
        firstLineIndent=0,
        spaceAfter=0
    )

    elems = []

    if layout["name"]:
        elems.append(Paragraph(layout["name"], name_style))
    if layout["contact_line"]:
        elems.append(Paragraph(layout["contact_line"], contact_style))

    for sec in layout["sections"]:
        elems.append(Paragraph(sec["title"], section_style))

        if sec["type"] == "summary":
            summary_text = " ".join(sec["lines"])
            elems.append(Paragraph(summary_text, body_style))

        elif sec["type"] == "skills":
            for row in sec["rows"]:
                if row["label"]:
                    text = f"<b>{row['label']}:</b> {', '.join(row['items'])}"
                else:
                    text = ", ".join(row["items"])
                elems.append(Paragraph(text, body_style))

        elif sec["type"] in ("experience", "education"):
            for entry in sec["entries"]:
                line1_left = entry.get("line1_left", "")
                line1_right = entry.get("line1_right", "")
                line2_left = entry.get("line2_left", "")
                line2_right = entry.get("line2_right", "")

                line1 = "    ".join([x for x in [line1_left, line1_right] if x])
                line2 = "    ".join([x for x in [line2_left, line2_right] if x])

                if line1:
                    elems.append(Paragraph(f"<b>{line1}</b>", body_style))
                if line2:
                    elems.append(Paragraph(line2, body_style))

                for b in entry["bullets"]:
                    elems.append(Paragraph(f"{bullet} {b}", bullet_style))

                if template.get("page_density") != "one_page_compact" and cfg["entry_space_after"] > 0:
                    elems.append(Spacer(1, cfg["entry_space_after"]))

        elif sec["type"] == "projects":
            for entry in sec["entries"]:
                line1_left = entry.get("line1_left", "")
                line1_right = entry.get("line1_right", "")
                line1 = "    ".join([x for x in [line1_left, line1_right] if x])

                if line1:
                    elems.append(Paragraph(f"<b>{line1}</b>", body_style))

                for b in entry["bullets"]:
                    elems.append(Paragraph(f"{bullet} {b}", bullet_style))

                if template.get("page_density") != "one_page_compact" and cfg["entry_space_after"] > 0:
                    elems.append(Spacer(1, cfg["entry_space_after"]))

        elif sec["type"] == "certifications":
            for b in sec["bullets"]:
                elems.append(Paragraph(f"{bullet} {b}", bullet_style))

    doc.build(elems)
    print(f"✅ PDF written → {filename}")
    return filename
# ---------------------------------------------
# 8) LaTeX exporter
# ---------------------------------------------
def export_to_latex(layout: Dict[str, Any], template: Dict[str, Any], filename: str = "aligned_resume.tex") -> str:
    cfg = style_profile(template)
    bullet = bullet_symbol(template)

    lines = []
    lines.append(r"\documentclass[10pt]{article}")
    lines.append(r"\usepackage[margin=" + str(cfg["margins_in"]) + r"in]{geometry}")
    lines.append(r"\usepackage{enumitem}")
    lines.append(r"\setlength{\parindent}{0pt}")
    lines.append(r"\setlength{\parskip}{0pt}")
    lines.append(r"\begin{document}")

    if layout["name"]:
        lines.append(r"\textbf{\Large " + layout["name"].replace("&", r"\&") + r"}\\[2pt]")
    if layout["contact_line"]:
        lines.append(layout["contact_line"].replace("&", r"\&") + r"\\[6pt]")

    for sec in layout["sections"]:
        lines.append(r"\textbf{" + sec["title"].replace("&", r"\&") + r"}\\[2pt]")

        if sec["type"] == "summary":
            summary_text = " ".join(sec["lines"])
            lines.append(summary_text.replace("&", r"\&") + r"\\")
            lines.append(r"\\[2pt]")

        elif sec["type"] == "skills":
            for row in sec["rows"]:
                if row["label"]:
                    lines.append(
                        r"\textbf{" + row["label"].replace("&", r"\&") + "}: "
                        + ", ".join(row["items"]).replace("&", r"\&") + r"\\"
                    )
                else:
                    lines.append(", ".join(row["items"]).replace("&", r"\&") + r"\\")
            lines.append(r"\\[2pt]")

        elif sec["type"] in ("experience", "education"):
            for entry in sec["entries"]:
                left1 = entry.get("line1_left", "").replace("&", r"\&")
                right1 = entry.get("line1_right", "").replace("&", r"\&")
                lines.append(r"\textbf{" + left1 + r"}" + (r"\hfill " + right1 if right1 else "") + r"\\")
                left2 = entry.get("line2_left", "").replace("&", r"\&")
                right2 = entry.get("line2_right", "").replace("&", r"\&")
                if left2 or right2:
                    lines.append(left2 + (r"\hfill " + right2 if right2 else "") + r"\\")
                lines.append(r"\begin{itemize}[leftmargin=" + str(cfg["bullet_indent_in"]) + r"in,itemsep=0pt]")
                for b in entry["bullets"]:
                    lines.append(r"\item " + b.replace("&", r"\&"))
                lines.append(r"\end{itemize}")

        elif sec["type"] == "projects":
            for entry in sec["entries"]:
                left1 = entry.get("line1_left", "").replace("&", r"\&")
                right1 = entry.get("line1_right", "").replace("&", r"\&")
                lines.append(r"\textbf{" + left1 + r"}" + (r"\hfill " + right1 if right1 else "") + r"\\")
                lines.append(r"\begin{itemize}[leftmargin=" + str(cfg["bullet_indent_in"]) + r"in,itemsep=0pt]")
                for b in entry["bullets"]:
                    lines.append(r"\item " + b.replace("&", r"\&"))
                lines.append(r"\end{itemize}")

        elif sec["type"] == "certifications":
            lines.append(r"\begin{itemize}[leftmargin=" + str(cfg["bullet_indent_in"]) + r"in,itemsep=0pt]")
            for b in sec["bullets"]:
                lines.append(r"\item " + b.replace("&", r"\&"))
            lines.append(r"\end{itemize}")

        lines.append(r"\\[2pt]")

    lines.append(r"\end{document}")

    with open(filename, "w") as f:
        f.write("\n".join(lines))

    print(f"✅ LaTeX written → {filename}")
    return filename

# ---------------------------------------------
# 9) Driver
# ---------------------------------------------
def run_block5_export(resume: StructuredResume, session_id: str) -> Dict[str, str]:
    template = load_template_profile(session_id)
    layout = build_layout(resume, template)

    print("\n=== TEMPLATE USED FOR EXPORT ===")
    print(json.dumps(template, indent=2))

    print("\n=== STYLE PROFILE USED FOR EXPORT ===")
    print(json.dumps(style_profile(template), indent=2))

    outputs = {"text": "", "docx": "", "pdf": "", "latex": ""}

    print("\nWhich formats do you want to generate?")
    print("Options: text, docx, pdf, latex")
    fmt_raw = input("Enter comma-separated list (e.g. text,docx,pdf):\n> ").strip().lower()
    chosen = {x.strip() for x in fmt_raw.split(",") if x.strip()}

    if "text" in chosen:
        txt = render_plaintext(layout, template)
        txt_path = "aligned_resume.txt"
        with open(txt_path, "w") as f:
            f.write(txt)
        outputs["text"] = txt_path
        print(f"✅ Plain text written → {txt_path}")

    if "docx" in chosen:
        outputs["docx"] = export_to_docx(layout, template, filename="aligned_resume.docx")

    if "pdf" in chosen:
        outputs["pdf"] = export_to_pdf(layout, template, filename="aligned_resume.pdf")

    if "latex" in chosen:
        outputs["latex"] = export_to_latex(layout, template, filename="aligned_resume.tex")

    print("\n=== BLOCK 5 COMPLETE — EXPORT SUMMARY ===")
    print(json.dumps(outputs, indent=2))
    return outputs

# ---------------------------------------------
# 10) Run Block 5
# ---------------------------------------------
print("\n\n=== RUNNING BLOCK 5: TEMPLATE-AWARE FINAL EXPORT ===")
block5_outputs = run_block5_export(final_resume_struct, profile.session_id)

# **What Layer 3 Achieves Overall**

By the end of Layer 3, the system has done all the important optimization work.

It has:

loaded structured resume and JD data
measured alignment
identified gaps
generated controlled ATS-aware suggestions
let the user selectively add new skills
let the user selectively refine summary, experience, and projects
merged all accepted changes into one final structured resume
prepared that resume for export in multiple formats

So overall, Layer 3 is the heart of the project.

It is where the system stops being just a parser or analyzer and becomes an intelligent resume optimization workflow.

It is also the part that best reflects the architecture your professor liked, because it breaks the LLM usage into structured stages instead of relying on one large prompt.